# V3 Benchmarks 2080

Section 7 of the paper A Subset Indexed Attention Cache for Exact Attention over Many Subsets reruns from this notebook and its H100 twin. The two notebooks hold the same cells. The machine name in cell 1 is the only configuration difference.

## Governing documents

- Frozen Benchmark Plan v2 with cache replay and masked attention replay, 2026-10-31. The manifest, the protocol, the status values, the reporting rules and the five dataset schemas. No row runs without a manifest entry, and no entry is added without a paper destination.
- Benchmark Tables and Figures v2. The eight tables, the figure and the numerical agreement paragraph, generated from the datasets by `V2_Benchmarks_Tables.ipynb`. No table or figure is written by hand.

## Counts per machine

| Item | This machine | Both machines |
| --- | --- | --- |
| Benchmark inputs | 18 | 18 |
| Benchmarks | 18 | 36 |
| Methods per benchmark | 17 | 17 |
| Method evaluations | 306 | 612 |
| Agreement checks | 288 | 576 |
| Preparation measurements | 270 | 540 |
| Execution calls | 1,504 on the 2080, 2,016 on the H100 | 3,520 |

Methods per benchmark: the reference combination, the eager cache, cache replay, the eight other eager direct combinations and the six masked attention replays. Agreement checks are one per method except the reference. Execution calls depend on how many methods reach timing on each machine.

## Implementation identity

- Cache. `_SubsetAttentionLevelsFn` in `set_attention_model.py` at d7a7e72a, level walk forward, closed form backward pass, no chunks. Called through `subset_attention_levels_exact_bwd`. Every pinned module's file SHA256 is checked at the start of the run and recorded with the results.
- Cache replay. The unchunked cache call captured once as a CUDA graph after three graph warmups and replayed. No grouped fallback.
- Direct combinations, 9 eager. Masked attention default backend, masked attention math backend and gathered attention, each at chunk size all, 4,096 and 1,024. Masked attention is `torch.nn.functional.scaled_dot_product_attention` with a boolean subset mask through `set_attention_bench.sdpa_chunk`. Gathered attention is `subset_attention_uncached`, indices built inside execution.
- Masked attention replays, 6. Each masked attention combination captured once as a CUDA graph and replayed, its whole P x n mask moved to the GPU once before capture. Gathered attention has no replay.
- Masked attention representation in eager mode. One P x n boolean mask built on the CPU in preparation. Each chunk's rows are transferred to the GPU during execution. Transfer time and memory count in execution.
- Precision. float32. TF32 off.
- Constants. C = 24, d_k = d_v = 80.
- Reference combination. Masked attention, math backend, chunk 1,024.

## Benchmark inputs

The 2080 creates the 18 authoritative input files once, hashed and saved as `.pkl.lz4`. The H100 loads those same files for every evaluation and never creates or replaces them. Each file holds the subsets, Q, K, V and the loss weights W. Training step inputs add the record, the model parameters, the optimizer state and the random number generator state.

Shared preparation is timed on each machine by regenerating the subsets and tensors from the seed, 5 repeats. The regenerated input is hash checked against the saved file once. That regeneration is a timing diagnostic and never writes an input file.

## Protocol per benchmark

1. Preparation. Shared preparation, cache own preparation and masked attention own preparation, 5 repeats each. Gathered attention has no preparation row.
2. Reference. The reference combination runs first. Its outputs and gradients must be finite. If not, the benchmark status is reference unavailable and nothing else runs in that benchmark. The 30 s ceiling applies to this call too.
3. Agreement. Each other method runs one agreement call against the reference, exactly as it is timed. Wall clock recorded.
4. Ceiling. An agreement call completing above 30 s gets status ceiling exceeded and no further calls. The call is never interrupted.
5. Replay verification. Before any timed call, each replay method's captured graph is replayed three times untimed and the later executions are compared with the first. A training step replay is also compared with the eager step of the same method from the same state. A failure ends the evaluation as failed agreement.
6. Timing. 2 untimed warmups, 5 timed repeats, median reported. Peak allocated and peak reserved GPU memory are reset before every call.
7. State restore. Training step benchmarks restore the model, optimizer and random number generator states before every reference, agreement, warmup and timed call.
8. Order. Reference, cache, cache replay, the other eager direct combinations in menu order, then the masked attention replays in menu order. Benchmarks run by ascending M, attention before training step.
9. Suite budget. 10 h operational wall clock on the H100. Unrun benchmarks keep their rows with a budget status.

## Outputs

This notebook saves one pickle, `section7_benchmarks_{machine}.pkl.lz4`, with five DataFrames, `benchmarks`, `method_evaluations`, `execution_calls`, `agreement_checks`, `preparation_measurements`, and a metadata dict with the manifest hash, the module hashes, the environment record, the tolerances, the timing boundaries and `mock = False`. `V2_Benchmarks_Tables.ipynb` joins the two machines' pickles on `benchmark_input_id`, verifies that both machines recorded the same input SHA256, and generates the eight tables, the paragraph and the figure from the combined 36 benchmark rows.ph and the figure from the combined 36 benchmark rows.

## Cell 1. Configuration, implementation identity and the manifest

Run in a fresh kernel. Configuration constants sit at the top. `MACHINE` selects the machine's paths from `PATHS`, and it is the only value that differs in the 2080 notebook. The cell puts `src` at the front of the import path, refuses to continue if any library module was already loaded from elsewhere, verifies that every imported module's file sits under `src`, and compares the SHA256 of the five library files with the pinned values. Three are the public v1.0 release at commit fc4ec03, `set_attention.py`, `set_attention_cache.py` and `set_attention_bench.py`. `set_attention_data.py` is that release with one change, `batch_ret_loss` accumulating the per subset loss in float64, made after the first suite found its float32 accumulation 2.5 times outside the loss tolerance at n = 496, M = 512,000. `set_attention_model.py` is that release with one change to the cache, the level walk's per level forward update and its three per level backward chains (child scaling, terminal fold, fact contribution) each run as one Triton kernel in place of the torch operations of the release; the mathematics, the gathers, the scatter reductions, the `index_add_` accumulations and the saved tensors are those of the release. The forward update kernel updates each state row in place and holds a barrier between its loads and its stores, so every warp of a program reads the row's maximum and normalizer before any warp overwrites them. Without that barrier repeated calls on the 2080 at M = 256,000 returned wrong outputs for whole runs in 6 to 9 of 10 calls; with it, 30 repeated calls at M = 256,000 and 10 grouped calls at M = 512,000 matched the masked attention reference in every output and gradient. A mismatch stops the notebook. The files must reach the H100 byte for byte, CRLF line endings included. Triton is required, `triton-windows` on the 2080 and the Triton bundled with the Linux torch wheel on the H100. TF32 is turned off. The cell does not write benchmark inputs or results. Python may write `__pycache__` on import.

The manifest has one row per benchmark input, 18 rows, in suite execution order, ascending M first and attention before training step within a budget. It carries every field of the `benchmarks` dataset that is fixed before generation. P, F, N, D and coverage are filled in cell 2 from the input files.

Subset selection maps to the library samplers as follows. Run sampler is `sample_subsets_powerset_runs` with run length λ = 10n. Fixed size run sampler is `sample_subsets_runs` with run length 8 and sizes 4, 8, 16, 32, 64. Individual sampler is `sample_subsets_powerset`. These three samplers are keyed by `SEED` and the row's `sampler_key`. Full power set is `sample_subsets_powerset_runs` with M = T, which returns every nonempty subset in canonical order with no random choice, so its `sampler_key` is null. null.

In [12]:
# Configuration
MACHINE = "2080"                                   # the H100 notebook sets "H100"
PATHS = {
    "2080": {"root": r"E:\apriori-ml-v3"},
    "H100": {"root": "/workspace/apriori-ml-v3"},  # set to the pod's checkout before the H100 run
}
SEED = 367
C, D_K, D_V = 24, 80, 80
PRECISION = "float32"
CEILING_MS = 30_000.0
SUITE_BUDGET_H = 10.0
WARMUPS, TIMED_REPEATS, PREP_REPEATS = 2, 5, 5
CHUNK_SIZES = {"all": None, "4096": 4096, "1024": 1024}
FIXED_RUN_LEN, FIXED_RUN_SIZES = 8, (4, 8, 16, 32, 64)
IMPLEMENTATION_COMMIT = "fc4ec03, with batch_ret_loss in set_attention_data.py accumulating the subset loss in float64, and the level walk's per level forward update and three backward chains in set_attention_model.py fused into Triton kernels, the forward update kernel with a barrier between its loads and its in place stores, the encoder's three per call input checks skippable through check_inputs and batch_ret_loss accepting a prebuilt counts tensor with a known output size, both so the training step can be captured in a CUDA graph (2026-10-29); the level walk's optional differentiable log normalizer output (return_lse), and the forest plan built with one stable sort by depth and sent to the GPU in one copy (2026-10-31)"
FROZEN_FILE_SHA256 = {                             # public v1.0 release, commit fc4ec03, src folder
    "set_attention.py":       "8580eb3ce583edec900c993d58793e98d2ef4e9c434f0dddc5a58991918f0d52",
    "set_attention_cache.py": "341de32d3617f1eeff291bcb9aadd700dbc726e8a9ac3d5b86456db98248f4c5",
    "set_attention_data.py":  "fd4c5a486477ccc30a7c202891ffbc579dc02b2bc5ae5988e94cbe6d7ac23cea",
    "set_attention_model.py": "d7a7e72a2a111bb40e45cbc61ed844349e51ed15c7c22749a817893eb8eee345",
    "set_attention_bench.py": "80b9a7eddb88ddee8948d997808643e476328e915b7ac09847e3ff25f64ea72a",
}

import os, sys, hashlib, time, platform
from math import comb
import numpy as np
import pandas as pd
import torch

ROOT = PATHS[MACHINE]["root"]
SRC_DIR = os.path.join(ROOT, "src")
INPUT_DIR = os.path.join(ROOT, "inputs")           # the 18 frozen benchmark input files
RESULT_DIR = os.path.join(ROOT, "results")

# Fresh kernel and import path
foreign = [m for m in sys.modules if m.startswith("set_attention")
           and not os.path.normcase(os.path.abspath(getattr(sys.modules[m], "__file__", "") or ""))
           .startswith(os.path.normcase(SRC_DIR) + os.sep)]
assert not foreign, f"library modules loaded from outside src, restart the kernel: {foreign}"
sys.path = [p for p in sys.path if os.path.normcase(os.path.abspath(p)) != os.path.normcase(SRC_DIR)]
sys.path.insert(0, SRC_DIR)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

IMPLEMENTATION_FILE_SHA256 = {f: sha256_file(os.path.join(SRC_DIR, f)) for f in FROZEN_FILE_SHA256}
V3_MODIFIED = {"set_attention_cache.py"}                 # v3 adds the run trie generator to this file, the other four stay frozen
for f, frozen in FROZEN_FILE_SHA256.items():
    if f not in V3_MODIFIED:
        assert IMPLEMENTATION_FILE_SHA256[f] == frozen, f"{f} differs from commit {IMPLEMENTATION_COMMIT}"

import set_attention, set_attention_cache, set_attention_data, set_attention_model, set_attention_bench
for mod in (set_attention, set_attention_cache, set_attention_data, set_attention_model, set_attention_bench):
    loaded = os.path.normcase(os.path.abspath(mod.__file__))
    assert loaded.startswith(os.path.normcase(SRC_DIR) + os.sep), f"{mod.__name__} loaded from {mod.__file__}"
from set_attention_cache import (sample_subsets_powerset, sample_subsets_runs,
                                 sample_subsets_powerset_runs, build_prefix_trie_canonical,
                                 powerset_total)
from set_attention_data import forest_level_plan_np, plan_to_device
from set_attention_model import subset_attention_levels_exact_bwd, subset_attention_uncached
from set_attention_bench import (make_inputs, make_weights, scores, sdpa_layout, sdpa_chunk,
                                 chunks_of)
assert set_attention_bench.RUN_LEN == FIXED_RUN_LEN and tuple(set_attention_bench.RUN_SIZES) == FIXED_RUN_SIZES

# Precision and device
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
assert torch.cuda.is_available(), "the benchmarks run on the GPU"
DEVICE = torch.device("cuda")
MACHINE_ENVIRONMENT = {
    "machine": MACHINE, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "python": platform.python_version(), "os": platform.platform(),
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32, "tf32_cudnn": torch.backends.cudnn.allow_tf32}

# Direct menu, in menu order
DIRECT_MENU = [("masked attention", "default", "all"), ("masked attention", "default", "4096"),
               ("masked attention", "default", "1024"), ("masked attention", "math", "all"),
               ("masked attention", "math", "4096"), ("masked attention", "math", "1024"),
               ("gathered attention", None, "all"), ("gathered attention", None, "4096"),
               ("gathered attention", None, "1024")]
REFERENCE_COMBINATION = ("masked attention", "math", "1024")

def combination_label(method, backend, chunk):
    head = {"masked attention": backend, "gathered attention": "gathered"}[method]
    return f"{head}@{chunk}"

# Manifest, the plan's Section 1 grid
def manifest_rows():
    rows = []
    def add(role, btype, n, M, selection, lam, destinations):
        T = powerset_total(n)
        bt = "attention" if btype == "attention benchmark" else "step"
        input_id = f"v2_{role.replace(' ', '_')}_{bt}_n{n}_M{M}"
        rows.append(dict(
            benchmark_input_id=input_id, benchmark_type=btype, subset_selection=selection,
            experimental_role=role, n=n, T=str(T), budget_M=int(M),
            requested_run_length_lambda=(int(lam) if lam is not None else None),
            full_power_set_ratio=T / (n * 2 ** (n - 1)), C=C, d_k=D_K, d_v=D_V,
            precision=PRECISION, seed=SEED,
            sampler_key=(None if selection == "full power set" else input_id),
            paper_destination=destinations))
    for n in (300, 496):
        for M in (64_000, 128_000, 256_000, 512_000):
            dest = ["tab:main_attention", "tab:prep_attention", "fig:speedup", "paragraph"]
            if M == 64_000:
                dest.append("tab:control")
            add("main comparison", "attention benchmark", n, M, "run sampler", 10 * n, dest)
        for M in (64_000, 512_000):
            add("main comparison", "training step benchmark", n, M, "run sampler", 10 * n,
                ["tab:main_step", "tab:prep_step", "fig:speedup", "paragraph"])
    for n in (14, 16):
        add("small record boundary", "attention benchmark", n, powerset_total(n), "full power set", None,
            ["tab:boundary"])
    for n in (300, 496):
        add("fixed size run comparison", "attention benchmark", n, 64_000, "fixed size run sampler", None,
            ["tab:boundary", "paragraph"])
        add("limited sharing control", "attention benchmark", n, 64_000, "individual sampler", None,
            ["tab:control"])
    return rows

ROLE_ORDER = {"main comparison": 0, "small record boundary": 1, "fixed size run comparison": 2,
              "limited sharing control": 3}
manifest = pd.DataFrame(manifest_rows())
manifest["type_order"] = (manifest.benchmark_type != "attention benchmark").astype(int)
manifest["role_order"] = manifest.experimental_role.map(ROLE_ORDER)
manifest = (manifest.sort_values(["budget_M", "type_order", "n", "role_order"])
            .drop(columns=["type_order", "role_order"]).reset_index(drop=True))
manifest["requested_run_length_lambda"] = manifest["requested_run_length_lambda"].astype("Int64")
manifest["execution_order"] = np.arange(1, len(manifest) + 1)
manifest["benchmark_id"] = manifest.benchmark_input_id + "@" + MACHINE
manifest["machine"] = MACHINE

assert len(manifest) == 18
assert manifest.benchmark_input_id.is_unique
assert (manifest.experimental_role == "main comparison").sum() == 12
print(MACHINE_ENVIRONMENT)
manifest[["execution_order", "benchmark_input_id", "benchmark_type", "experimental_role",
          "subset_selection", "n", "budget_M", "requested_run_length_lambda", "paper_destination"]]

{'machine': '2080', 'gpu': 'NVIDIA GeForce RTX 2080 SUPER', 'torch': '2.5.1+cu121', 'cuda': '12.1', 'cudnn': 90100, 'python': '3.12.7', 'os': 'Windows-10-10.0.19045-SP0', 'tf32_matmul': False, 'tf32_cudnn': False}


,execution_order,benchmark_input_id,benchmark_type,experimental_role,subset_selection,n,budget_M,requested_run_length_lambda,paper_destination
0,1,v2_small_record_boundary_attention_n14_M16383,attention benchmark,small record boundary,full power set,14,16383,<NA>,[tab:boundary]
1,2,v2_main_comparison_attention_n300_M64000,attention benchmark,main comparison,run sampler,300,64000,3000,"[tab:main_attention, tab:prep_attention, fig:s..."
2,3,v2_fixed_size_run_comparison_attention_n300_M6...,attention benchmark,fixed size run comparison,fixed size run sampler,300,64000,<NA>,"[tab:boundary, paragraph]"
3,4,v2_limited_sharing_control_attention_n300_M64000,attention benchmark,limited sharing control,individual sampler,300,64000,<NA>,[tab:control]
4,5,v2_main_comparison_attention_n496_M64000,attention benchmark,main comparison,run sampler,496,64000,4960,"[tab:main_attention, tab:prep_attention, fig:s..."
5,6,v2_fixed_size_run_comparison_attention_n496_M6...,attention benchmark,fixed size run comparison,fixed size run sampler,496,64000,<NA>,"[tab:boundary, paragraph]"
6,7,v2_limited_sharing_control_attention_n496_M64000,attention benchmark,limited sharing control,individual sampler,496,64000,<NA>,[tab:control]
7,8,v2_main_comparison_step_n300_M64000,training step benchmark,main comparison,run sampler,300,64000,3000,"[tab:main_step, tab:prep_step, fig:speedup, pa..."
8,9,v2_main_comparison_step_n496_M64000,training step benchmark,main comparison,run sampler,496,64000,4960,"[tab:main_step, tab:prep_step, fig:speedup, pa..."
9,10,v2_small_record_boundary_attention_n16_M65535,attention benchmark,small record boundary,full power set,16,65535,<NA>,[tab:boundary]


## Cell 2. Generate the 14 attention benchmark inputs, verify every input file that exists

One file per benchmark input, `inputs\<benchmark_input_id>.pkl.lz4`. Before any work the cell lists which of the 18 files exist. An existing file of either kind is never rewritten. It is loaded, its content hash is recomputed from the stored arrays and compared with the value stored inside it, and its statistics are taken from it into the manifest. A missing attention file is built here. A missing training step file is left to cell 3, which runs only on the 2080, and the cell reports it as pending. With all 18 files present, as on the H100 and on a 2080 rerun, this cell completes the manifest and cell 3 is not needed.

Each attention file holds the manifest fields, the subsets as offsets and a flat int16 array in canonical order, Q, K, V in float32, the SHA256 of the loss weights W, and the trie statistics P, F, N, D, coverage and reuse ratio. W is regenerated from the seed at load time by `make_weights` and must match the stored SHA256 or the benchmark stops. Two hashes describe a file. `benchmark_input_sha256` is the SHA256 of the file bytes and covers transfer to the H100. `content_sha256` is the SHA256 over the subset arrays, Q, K, V and the W hash in that order, and it is what the input generation check on each machine compares against. A training step file is verified through its own `subsets_sha256` and `step_content_sha256`, defined here as in cell 3. The file's creation time and machine are outside the content hash.

Subsets come from the samplers named in cell 1, sorted into canonical order and checked for duplicates exactly as `set_attention_bench.sample_set` does. For the run sampler the subset count is asserted equal to R × L from `powerset_run_base`. For the full power set it is asserted equal to T. The trie is built with `build_prefix_trie_canonical` to record N and D only. The benchmark cells produce it again inside cache own preparation, generated from the run descriptor on run sampler and full power set rows and built from the subset list on the others.

In [13]:
from set_attention_cache import powerset_run_base
import apriori.serialization as _ser
assert os.path.normcase(os.path.abspath(_ser.__file__)).startswith(os.path.normcase(SRC_DIR) + os.sep), _ser.__file__
from apriori.serialization import save_to_pickle, load_from_pickle

os.makedirs(INPUT_DIR, exist_ok=True)
INPUT_FIELDS = ["benchmark_input_id", "benchmark_type", "subset_selection", "experimental_role", "n", "T",
                "budget_M", "requested_run_length_lambda", "full_power_set_ratio", "C", "d_k", "d_v",
                "precision", "seed", "sampler_key", "paper_destination"]

def input_path(input_id):
    return os.path.join(INPUT_DIR, f"{input_id}.pkl.lz4")

def sha256_array(a):
    a = np.ascontiguousarray(a)
    return hashlib.sha256(memoryview(a).cast("B")).hexdigest()

def content_sha256(indptr, flat, q, K, V, W_sha):
    """The logical input hash: subset arrays, Q, K, V and the W hash, in this order."""
    h = hashlib.sha256()
    for a in (np.ascontiguousarray(indptr, dtype=np.int64), np.ascontiguousarray(flat, dtype=np.int16),
              q.numpy(), K.numpy(), V.numpy()):
        h.update(memoryview(np.ascontiguousarray(a)).cast("B"))
    h.update(W_sha.encode("ascii"))
    return h.hexdigest()

def generate_subsets(row):
    """The row's subsets as sorted tuples in canonical order, from the sampler the plan names."""
    n, M, sel, key = int(row.n), int(row.budget_M), row.subset_selection, row.sampler_key
    if sel == "run sampler":
        s = sample_subsets_powerset_runs(n, key, M, int(row.requested_run_length_lambda), seed=SEED, epoch=0)
    elif sel == "fixed size run sampler":
        s = sample_subsets_runs(n, key, M, FIXED_RUN_LEN, k_sizes=FIXED_RUN_SIZES, seed=SEED, epoch=0)
    elif sel == "individual sampler":
        s = sample_subsets_powerset(n, key, M, seed=SEED, epoch=0)
    elif sel == "full power set":
        s = sample_subsets_powerset_runs(n, "", M, 1, seed=SEED, epoch=0)   # M = T returns every subset
    else:
        raise ValueError(sel)
    s = sorted(tuple(int(x) for x in S) for S in s)
    if len(set(s)) != len(s):
        raise AssertionError("duplicate subsets")
    if sel == "run sampler":
        _, R, L, _ = powerset_run_base(n, M, int(row.requested_run_length_lambda))
        assert len(s) == R * L, (len(s), R * L)
    if sel == "full power set":
        assert len(s) == powerset_total(n), (len(s), powerset_total(n))
    return s

def flatten_subsets(subsets):
    lens = np.fromiter((len(S) for S in subsets), dtype=np.int64, count=len(subsets))
    indptr = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)
    flat = np.fromiter((x for S in subsets for x in S), dtype=np.int16, count=int(lens.sum()))
    return indptr, flat

def generate_tensors(n, P):
    """Q, K, V and the W hash from the seed, as every machine regenerates them."""
    q, K, V = [t.detach() for t in make_inputs(n, C, D_K, SEED, torch.float32, "cpu")]
    W = make_weights(P, C, D_K, SEED, torch.float32, "cpu")
    W_sha = sha256_array(W.numpy())
    del W
    return q, K, V, W_sha

def build_input_file(row, path):
    t0 = time.perf_counter()
    subsets = generate_subsets(row)
    P, F = len(subsets), sum(len(S) for S in subsets)
    trie = build_prefix_trie_canonical(subsets)
    N, D = int(len(trie["pos"])), int(trie["depth"].max()) + 1
    indptr, flat = flatten_subsets(subsets)
    del subsets, trie
    q, K, V, W_sha = generate_tensors(int(row.n), P)
    T = int(row["T"])
    rec = {k: row[k] for k in INPUT_FIELDS}
    rec["requested_run_length_lambda"] = (None if pd.isna(rec["requested_run_length_lambda"])
                                          else int(rec["requested_run_length_lambda"]))
    rec.update(P=P, F=F, N=N, D=D, coverage=P / T, reuse_ratio=N / F,
               subset_indptr=indptr, subset_flat=flat, q=q, K=K, V=V, W_sha256=W_sha, W_seed=SEED,
               content_sha256=content_sha256(indptr, flat, q, K, V, W_sha),
               implementation_commit=IMPLEMENTATION_COMMIT, implementation_file_sha256=IMPLEMENTATION_FILE_SHA256,
               created=time.strftime("%Y-%m-%d %H:%M:%S"), created_on=MACHINE)
    save_to_pickle(rec, path, lz4_compression=True)
    return dict(benchmark_input_id=row.benchmark_input_id, P=P, F=F, N=N, D=D, coverage=P / T,
                reuse_ratio=N / F, content_sha256=rec["content_sha256"], benchmark_input_sha256=sha256_file(path),
                file_mib=os.path.getsize(path) / 2 ** 20, seconds=time.perf_counter() - t0, action="built")

def hash_tree(h, o):
    """Feed a nested structure of tensors, arrays, dicts, lists and scalars to a hash in a fixed order."""
    if torch.is_tensor(o):
        a = o.detach().cpu().contiguous().numpy() if o.dtype != torch.bfloat16 else o.detach().cpu().float().numpy()
        h.update(f"tensor:{a.dtype}:{a.shape}".encode()); h.update(memoryview(np.ascontiguousarray(a)).cast("B"))
    elif isinstance(o, np.ndarray):
        h.update(f"array:{o.dtype}:{o.shape}".encode()); h.update(memoryview(np.ascontiguousarray(o)).cast("B"))
    elif isinstance(o, dict):
        for k in sorted(o, key=str):
            h.update(f"key:{k}".encode()); hash_tree(h, o[k])
    elif isinstance(o, (list, tuple)):
        h.update(f"seq:{len(o)}".encode())
        for v in o:
            hash_tree(h, v)
    else:
        h.update(f"val:{o!r}".encode())

def subsets_sha256(indptr, flat):
    h = hashlib.sha256()
    hash_tree(h, [np.ascontiguousarray(indptr, dtype=np.int64), np.ascontiguousarray(flat, dtype=np.int16)])
    return h.hexdigest()

def step_content_sha256(indptr, flat, record, snap):
    """The training step file's logical hash: subset arrays, record, model, optimizer and RNG states."""
    h = hashlib.sha256()
    hash_tree(h, [np.ascontiguousarray(indptr, dtype=np.int64), np.ascontiguousarray(flat, dtype=np.int16),
                  record["cols"], record["vals"], record["name_vectors"],
                  snap["enc"], snap["stack"], snap["opt"], snap["rng_cpu"], snap["rng_cuda"]])
    return h.hexdigest()

def verify_input_file(row, path):
    """An existing file of either kind: recompute its content hash from the stored arrays and compare."""
    t0 = time.perf_counter()
    rec = load_from_pickle(path)
    assert rec["benchmark_input_id"] == row.benchmark_input_id, path
    for k in ["n", "budget_M", "subset_selection", "experimental_role", "benchmark_type"]:
        assert rec[k] == row[k], (k, rec[k], row[k])
    if row.benchmark_type == "training step benchmark":
        assert subsets_sha256(rec["subset_indptr"], rec["subset_flat"]) == rec["subsets_sha256"], path
        got = step_content_sha256(rec["subset_indptr"], rec["subset_flat"], rec["record"], rec["state"])
    else:
        got = content_sha256(rec["subset_indptr"], rec["subset_flat"], rec["q"], rec["K"], rec["V"], rec["W_sha256"])
    assert got == rec["content_sha256"], f"content hash mismatch in {path}"
    return dict(benchmark_input_id=row.benchmark_input_id, P=rec["P"], F=rec["F"], N=rec["N"], D=rec["D"],
                coverage=rec["coverage"], reuse_ratio=rec["reuse_ratio"], content_sha256=rec["content_sha256"],
                benchmark_input_sha256=sha256_file(path), file_mib=os.path.getsize(path) / 2 ** 20,
                seconds=time.perf_counter() - t0, action="verified")

existing = {r.benchmark_input_id: os.path.exists(input_path(r.benchmark_input_id)) for _, r in manifest.iterrows()}
to_build = [r.benchmark_input_id for _, r in manifest.iterrows()
            if not existing[r.benchmark_input_id] and r.benchmark_type == "attention benchmark"]
pending = [r.benchmark_input_id for _, r in manifest.iterrows()
           if not existing[r.benchmark_input_id] and r.benchmark_type == "training step benchmark"]
print(f"{sum(existing.values())} of 18 input files exist, {len(to_build)} attention files to build, "
      f"{len(pending)} training step files pending cell 3")

generated = []
for _, row in manifest.iterrows():
    path = input_path(row.benchmark_input_id)
    if existing[row.benchmark_input_id]:
        r = verify_input_file(row, path)
    elif row.benchmark_type == "attention benchmark":
        r = build_input_file(row, path)
    else:
        continue
    generated.append(r)
    print(f"{r['action']:8s} {r['benchmark_input_id']:55s} P={r['P']:>9,} F={r['F']:>12,} N={r['N']:>12,} "
          f"D={r['D']:>4} {r['file_mib']:8.1f} MiB {r['seconds']:7.1f} s")

generated = pd.DataFrame(generated)
manifest = manifest.drop(columns=[c for c in generated.columns if c != "benchmark_input_id" and c in manifest.columns])
manifest = manifest.merge(generated, on="benchmark_input_id", how="left")
for c in ("P", "F", "N", "D"):
    manifest[c] = pd.to_numeric(manifest[c]).astype("Int64")
if not pending:
    assert manifest.benchmark_input_sha256.notna().all() and len(manifest) == 18
manifest[["execution_order", "benchmark_input_id", "P", "F", "N", "D", "reuse_ratio", "coverage",
          "benchmark_input_sha256", "file_mib", "seconds", "action"]]

18 of 18 input files exist, 0 attention files to build, 0 training step files pending cell 3
verified v2_small_record_boundary_attention_n14_M16383           P=   16,383 F=     114,688 N=      16,383 D=  14      0.3 MiB     0.0 s
verified v2_main_comparison_attention_n300_M64000                P=   63,000 F=   9,462,294 N=      66,107 D= 172      5.0 MiB     0.1 s
verified v2_fixed_size_run_comparison_attention_n300_M64000      P=   64,000 F=   1,587,200 N=     246,307 D=  64      5.2 MiB     0.0 s
verified v2_limited_sharing_control_attention_n300_M64000        P=   64,000 F=   9,603,002 N=   9,154,745 D= 173      8.9 MiB     0.1 s
verified v2_main_comparison_attention_n496_M64000                P=   59,520 F=  14,677,812 N=      62,468 D= 264      7.9 MiB     0.1 s
verified v2_fixed_size_run_comparison_attention_n496_M64000      P=   64,000 F=   1,587,200 N=     246,208 D=  64      8.1 MiB     0.0 s
verified v2_limited_sharing_control_attention_n496_M64000        P=   64,000 F=  15,8

,execution_order,benchmark_input_id,P,F,N,D,reuse_ratio,coverage,benchmark_input_sha256,file_mib,seconds,action
0,1,v2_small_record_boundary_attention_n14_M16383,16383,114688,16383,14,0.142848,1.000000e+00,cb9249fe03bf6f85fd05841fc885959a3f2ef70f15045f...,0.336365,0.003329,verified
1,2,v2_main_comparison_attention_n300_M64000,63000,9462294,66107,172,0.006986,3.092729e-86,31f07a0a50e563ab6aa7ad229385796feaadd6a56c5ebe...,5.039701,0.052933,verified
2,3,v2_fixed_size_run_comparison_attention_n300_M6...,64000,1587200,246307,64,0.155183,3.141820e-86,2c465e03354342bb41455e6fbcfdda4e3e07c043cf757b...,5.246055,0.035184,verified
3,4,v2_limited_sharing_control_attention_n300_M64000,64000,9603002,9154745,173,0.953321,3.141820e-86,1d8c93617f2806604694c75d2c1bee022cf6d0586cb919...,8.910215,0.071954,verified
4,5,v2_main_comparison_attention_n496_M64000,59520,14677812,62468,264,0.004256,2.909277e-145,95bbb5275cad59f964561947b777065f6a224cf4f8cbd3...,7.919517,0.085321,verified
5,6,v2_fixed_size_run_comparison_attention_n496_M6...,64000,1587200,246208,64,0.155121,3.128255e-145,1f854c8f5390551002dae75167e71f7981e4a95ecd06be...,8.142472,0.046565,verified
6,7,v2_limited_sharing_control_attention_n496_M64000,64000,15877122,15430401,286,0.971864,3.128255e-145,85a2bd747a0e034b32643015043e7bc50b27def4d4f396...,15.300050,0.108904,verified
7,8,v2_main_comparison_step_n300_M64000,63000,9431510,66105,173,0.007009,3.092729e-86,9a7d82c7847899ea9bb0c19810f98e463f4cf2690f11e6...,85.070204,0.552011,verified
8,9,v2_main_comparison_step_n496_M64000,59520,14778502,62470,260,0.004227,2.909277e-145,7959adb2034a44912ea8abb662aa545e09c907c51dfde9...,85.365689,0.577253,verified
9,10,v2_small_record_boundary_attention_n16_M65535,65535,524288,65535,16,0.124998,1.000000e+00,7680106e1ab4bb834380305677f2e74f52c966ff6180ab...,0.738225,0.006916,verified


## Cell 3. Generate and save the 4 training step benchmark inputs

Export cell. It imports `GaapFeaturesML` from BB_Pipeline_v15 to read the two records, so it runs only on the 2080. The H100 loads the files it writes. Written once, never overwritten, verified on rerun as in cell 2. When it finishes, the cell unloads the BB_Pipeline_v15 `apriori` package, reimports `apriori.serialization` from `src`, and checks that the package on the path is the `src` one again.

Each file holds the manifest fields, the subsets as in cell 2, and the training step state. The record is the training period filing (2019 to 2024 Q2, at least 4 facts) with the nearest fact count to n, ties to the lowest matrix row, every fact kept, stored as `cols`, `vals` and the record's name vectors from the concept name embedding store. The count must equal the manifest's n. The model is `SemanticMaskedSetEncoder` loaded from `set_encoder_tv_sem.pt` with `SubsetInteractionStack` of 1 query and hidden 256, seeded 367, in training mode. The optimizer is AdamW, learning rate 1e-4 on the stack and 1e-5 on the encoder, weight decay 0.01, initialized by one untimed full set step at target 0. The frozen state is the encoder, stack and optimizer state dicts and the CPU and CUDA random number generator states taken after that step. Every reference, agreement, warmup and timed call on either machine restores that state from the file. It is never recreated on the H100.

Two hashes inside each file. `subsets_sha256` covers the subset arrays only, and it is what the shared preparation regeneration check on each machine compares against for training step rows. `content_sha256` covers the subset arrays, `cols`, `vals`, the name vectors, the encoder and stack state tensors, the optimizer state and the two random number generator states, and it is checked on every load. The checkpoint's SHA256 and the embedding store's name hash are recorded in the file. The subsets are keyed by `benchmark_input_id`, so they are new samples, not V2C4's.

In [8]:
# Export cell configuration
BB_DIR = r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"
SEM_CKPT = r"E:\apriori-ml\checkpoints\set_encoder_tv_sem.pt"
STORE_PATH = r"E:\ML\GaapFFS\concept_name_embeddings.pkl.lz4"
WINDOW_START, WINDOW_END = "2019-01-01", "2025-12-31"
MIN_FACTS = 4
LR, ENC_SCALE, WD = 1e-4, 0.1, 0.01
N_QUERIES, HIDDEN, Y_TARGET = 1, 256, 0.0
TOL_LOSS, TOL_GRAD, TOL_ZERO = (1e-6, 1e-4), 1e-3, 1e-6

import gc, copy, importlib
import torch.nn.functional as F
from set_attention_model import (SetModelConfig, SemanticMaskedSetEncoder, SubsetInteractionStack,
                                 encoder_layer_kv)

def unload_apriori():
    for _m in [k for k in sys.modules if k == "apriori" or k.startswith("apriori.")]:
        del sys.modules[_m]

def path_without(*dirs):
    drop = {os.path.normcase(os.path.abspath(d)) for d in dirs}
    return [p for p in sys.path if os.path.normcase(os.path.abspath(p)) not in drop]

# Apriori import for the records: BB_Pipeline_v15 first on the path for this cell only
os.environ["ENV"] = "prod"
unload_apriori()
sys.path = [BB_DIR] + path_without(BB_DIR)
from apriori.ml.features.gaap_features_ml import GaapFeaturesML

def in_train(fn):                                         # training period: 2019 to 2024 Q2
    y, q = fn[:4], fn[5:9]
    return y in ("2019", "2020", "2021", "2022", "2023") or (y == "2024" and q in ("QTR1", "QTR2"))

def tree_cpu(o):
    """Deep copy with every tensor detached, cloned and moved to the CPU."""
    if torch.is_tensor(o):
        return o.detach().cpu().clone()
    if isinstance(o, dict):
        return {k: tree_cpu(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return type(o)(tree_cpu(v) for v in o)
    return copy.deepcopy(o)

def hash_tree(h, o):
    """Feed a nested structure of tensors, arrays, dicts, lists and scalars to a hash in a fixed order."""
    if torch.is_tensor(o):
        a = o.detach().cpu().contiguous().numpy() if o.dtype != torch.bfloat16 else o.detach().cpu().float().numpy()
        h.update(f"tensor:{a.dtype}:{a.shape}".encode()); h.update(memoryview(np.ascontiguousarray(a)).cast("B"))
    elif isinstance(o, np.ndarray):
        h.update(f"array:{o.dtype}:{o.shape}".encode()); h.update(memoryview(np.ascontiguousarray(o)).cast("B"))
    elif isinstance(o, dict):
        for k in sorted(o, key=str):
            h.update(f"key:{k}".encode()); hash_tree(h, o[k])
    elif isinstance(o, (list, tuple)):
        h.update(f"seq:{len(o)}".encode())
        for v in o:
            hash_tree(h, v)
    else:
        h.update(f"val:{o!r}".encode())

def subsets_sha256(indptr, flat):
    h = hashlib.sha256()
    hash_tree(h, [np.ascontiguousarray(indptr, dtype=np.int64), np.ascontiguousarray(flat, dtype=np.int16)])
    return h.hexdigest()

def step_content_sha256(indptr, flat, record, snap):
    h = hashlib.sha256()
    hash_tree(h, [np.ascontiguousarray(indptr, dtype=np.int64), np.ascontiguousarray(flat, dtype=np.int16),
                  record["cols"], record["vals"], record["name_vectors"],
                  snap["enc"], snap["stack"], snap["opt"], snap["rng_cpu"], snap["rng_cuda"]])
    return h.hexdigest()

# ----- records -----
gml = GaapFeaturesML(start_date=WINDOW_START, end_date=WINDOW_END)
gml.assemble_window()
mat = gml.matrix
assert mat.has_canonical_format
nnz = np.asarray(mat.getnnz(axis=1)).ravel()
name_store = load_from_pickle(STORE_PATH)
names_fp = hashlib.sha256("|".join(np.asarray(gml.feature_names_by_col, dtype=object)).encode()).hexdigest()[:16]
assert name_store["smoke"] is False and name_store["names_sha256_16"] == names_fp
name_table = name_store["embeddings"]
assert name_table.shape == (mat.shape[1], 384)
train_rows = np.array([r for r in range(mat.shape[0]) if in_train(gml.file_names_by_row[r])])
tr4 = train_rows[nnz[train_rows] >= MIN_FACTS]

def pick_record(n_target):
    r = int(tr4[np.argmin(np.abs(nnz[tr4] - n_target))])
    sl = slice(mat.indptr[r], mat.indptr[r + 1])
    cols, vals = mat.indices[sl].astype(np.int64), mat.data[sl].astype(np.float32)
    assert np.all(np.diff(cols) > 0) and np.isfinite(vals).all()
    nv = np.ascontiguousarray(name_table[cols], dtype=np.float32)
    assert np.isfinite(nv).all()
    assert len(cols) == n_target, f"nearest training record has {len(cols)} facts, manifest n is {n_target}"
    return {"row": r, "file_name": gml.file_names_by_row[r], "n": int(len(cols)), "cols": cols, "vals": vals,
            "name_vectors": nv}

records = {n_t: pick_record(n_t) for n_t in (300, 496)}
del gml, mat, nnz, name_store, name_table, train_rows, tr4
gc.collect()

# ----- model, optimizer and the frozen state, once per record -----
ck = torch.load(SEM_CKPT, map_location="cpu", weights_only=False)
cfg_sem = SetModelConfig(**ck["config"])
ckpt_sha = sha256_file(SEM_CKPT)
assert cfg_sem.n_layers * cfg_sem.n_heads * N_QUERIES == C and cfg_sem.dim // cfg_sem.n_heads == D_K

def build_models(dev):
    torch.manual_seed(SEED)
    enc = SemanticMaskedSetEncoder(cfg_sem)
    enc.load_state_dict(ck["model"])
    stack = SubsetInteractionStack(cfg_sem.n_layers, cfg_sem.n_heads, cfg_sem.dim // cfg_sem.n_heads, N_QUERIES, HIDDEN)
    return enc.to(dev, torch.float32), stack.to(dev, torch.float32)

def record_tensors(rec, dev):
    n = rec["n"]
    return {"nv": torch.from_numpy(rec["name_vectors"]).to(dev), "vals": torch.from_numpy(rec["vals"]).to(dev),
            "pad": torch.zeros((1, n), dtype=torch.bool, device=dev),
            "y": torch.full((1,), Y_TARGET, dtype=torch.float32, device=dev)}

def upstream(enc, stack, ft):
    """Encoder, per layer keys and values, scores and values, full set prediction, as V2C4."""
    kv = encoder_layer_kv(enc, ft["nv"][None], ft["vals"][None], ft["pad"])
    z, V = stack.scores_values([(K[0], Vv[0]) for K, Vv in kv])
    n, Cc, dh = V.shape
    w = torch.softmax(z.view(1, n, Cc).masked_fill(ft["pad"][..., None], float("-inf")), dim=1)
    pf = stack.predict(torch.einsum("blc,blcd->bcd", w, V.view(1, n, Cc, dh)))
    return kv, z, V, pf

def frozen_state(rec):
    """The state every training step call restores: taken after one untimed full set AdamW step."""
    enc, stack = build_models(DEVICE)
    enc.train(); stack.train()
    opt = torch.optim.AdamW([{"params": enc.parameters(), "lr": LR * ENC_SCALE},
                             {"params": stack.parameters(), "lr": LR}], weight_decay=WD)
    ft = record_tensors(rec, DEVICE)
    torch.manual_seed(SEED)
    opt.zero_grad(set_to_none=True)
    F.huber_loss(upstream(enc, stack, ft)[3], ft["y"], delta=1.0).backward()
    opt.step()
    snap = {"enc": tree_cpu(enc.state_dict()), "stack": tree_cpu(stack.state_dict()),
            "opt": tree_cpu(opt.state_dict()), "rng_cpu": torch.get_rng_state().clone(),
            "rng_cuda": torch.cuda.get_rng_state().clone()}
    del enc, stack, opt, ft
    gc.collect(); torch.cuda.empty_cache()
    return snap

snaps = {n_t: frozen_state(rec) for n_t, rec in records.items()}

# ----- the 4 files -----
def build_step_file(row, path):
    t0 = time.perf_counter()
    rec, snap = records[int(row.n)], snaps[int(row.n)]
    subsets = generate_subsets(row)
    P, F_ = len(subsets), sum(len(S) for S in subsets)
    trie = build_prefix_trie_canonical(subsets)
    N, D = int(len(trie["pos"])), int(trie["depth"].max()) + 1
    indptr, flat = flatten_subsets(subsets)
    del subsets, trie
    T = int(row["T"])
    out = {k: row[k] for k in INPUT_FIELDS}
    out["requested_run_length_lambda"] = int(out["requested_run_length_lambda"])
    out.update(P=P, F=F_, N=N, D=D, coverage=P / T, reuse_ratio=N / F_,
               subset_indptr=indptr, subset_flat=flat, subsets_sha256=subsets_sha256(indptr, flat),
               record={k: rec[k] for k in ("row", "file_name", "n", "cols", "vals", "name_vectors")},
               model_config=dict(ck["config"]), checkpoint_path=SEM_CKPT, checkpoint_sha256=ckpt_sha,
               names_sha256_16=names_fp, n_queries=N_QUERIES, hidden=HIDDEN, y_target=Y_TARGET,
               lr=LR, enc_scale=ENC_SCALE, weight_decay=WD, tolerances=dict(loss=TOL_LOSS, grad=TOL_GRAD, zero=TOL_ZERO),
               state=snap, content_sha256=step_content_sha256(indptr, flat, rec, snap),
               implementation_commit=IMPLEMENTATION_COMMIT, implementation_file_sha256=IMPLEMENTATION_FILE_SHA256,
               created=time.strftime("%Y-%m-%d %H:%M:%S"), created_on=MACHINE)
    save_to_pickle(out, path, lz4_compression=True)
    return dict(benchmark_input_id=row.benchmark_input_id, P=P, F=F_, N=N, D=D, coverage=P / T,
                reuse_ratio=N / F_, content_sha256=out["content_sha256"], benchmark_input_sha256=sha256_file(path),
                file_mib=os.path.getsize(path) / 2 ** 20, seconds=time.perf_counter() - t0, action="built",
                record=rec["file_name"])

def verify_step_file(row, path):
    t0 = time.perf_counter()
    rec = load_from_pickle(path)
    assert rec["benchmark_input_id"] == row.benchmark_input_id, path
    assert subsets_sha256(rec["subset_indptr"], rec["subset_flat"]) == rec["subsets_sha256"], path
    got = step_content_sha256(rec["subset_indptr"], rec["subset_flat"], rec["record"], rec["state"])
    assert got == rec["content_sha256"], f"content hash mismatch in {path}"
    return dict(benchmark_input_id=row.benchmark_input_id, P=rec["P"], F=rec["F"], N=rec["N"], D=rec["D"],
                coverage=rec["coverage"], reuse_ratio=rec["reuse_ratio"], content_sha256=rec["content_sha256"],
                benchmark_input_sha256=sha256_file(path), file_mib=os.path.getsize(path) / 2 ** 20,
                seconds=time.perf_counter() - t0, action="verified", record=rec["record"]["file_name"])

step_rows = manifest[manifest.benchmark_type == "training step benchmark"]
built = []
for _, row in step_rows.iterrows():
    path = input_path(row.benchmark_input_id)
    r = verify_step_file(row, path) if os.path.exists(path) else build_step_file(row, path)
    built.append(r)
    print(f"{r['action']:8s} {r['benchmark_input_id']:40s} {r['record']:42s} P={r['P']:>9,} F={r['F']:>12,} "
          f"N={r['N']:>9,} D={r['D']:>4} {r['file_mib']:8.1f} MiB {r['seconds']:7.1f} s")

built = pd.DataFrame(built).drop(columns=["record"]).set_index("benchmark_input_id")
for c in built.columns:
    manifest.loc[manifest.benchmark_input_id.isin(built.index), c] = \
        built.loc[manifest.benchmark_input_id[manifest.benchmark_input_id.isin(built.index)], c].values
for c in ("P", "F", "N", "D"):
    manifest[c] = manifest[c].astype("Int64")
assert manifest.benchmark_input_sha256.notna().all() and len(manifest) == 18

# Apriori package back to the src copy, BB_Pipeline_v15 off the path
del records, snaps, ck
gc.collect(); torch.cuda.empty_cache()
unload_apriori()
sys.path = [SRC_DIR] + path_without(SRC_DIR, BB_DIR)
import apriori
import apriori.serialization as _ser
from apriori.serialization import save_to_pickle, load_from_pickle
assert os.path.normcase(os.path.abspath(apriori.__file__)).startswith(os.path.normcase(SRC_DIR) + os.sep), apriori.__file__
assert os.path.normcase(os.path.abspath(_ser.__file__)).startswith(os.path.normcase(SRC_DIR) + os.sep), _ser.__file__
manifest[["execution_order", "benchmark_input_id", "P", "F", "N", "D", "reuse_ratio", "benchmark_input_sha256", "file_mib", "action"]]

verified v2_main_comparison_step_n300_M64000      2019_QTR2_701347_0000701347-19-000022      P=   63,000 F=   9,431,510 N=   66,105 D= 173     85.1 MiB     0.4 s
verified v2_main_comparison_step_n496_M64000      2019_QTR2_763901_0001193125-19-144133      P=   59,520 F=  14,778,502 N=   62,470 D= 260     85.4 MiB     0.4 s
verified v2_main_comparison_step_n300_M512000     2019_QTR2_701347_0000701347-19-000022      P=  510,000 F=  76,490,624 N=  534,835 D= 180     89.6 MiB     0.6 s
verified v2_main_comparison_step_n496_M512000     2019_QTR2_763901_0001193125-19-144133      P=  510,880 F= 126,771,882 N=  536,094 D= 280     90.3 MiB     0.8 s


,execution_order,benchmark_input_id,P,F,N,D,reuse_ratio,benchmark_input_sha256,file_mib,action
0,1,v2_small_record_boundary_attention_n14_M16383,16383,114688,16383,14,0.142848,cb9249fe03bf6f85fd05841fc885959a3f2ef70f15045f...,0.336365,verified
1,2,v2_main_comparison_attention_n300_M64000,63000,9462294,66107,172,0.006986,31f07a0a50e563ab6aa7ad229385796feaadd6a56c5ebe...,5.039701,verified
2,3,v2_fixed_size_run_comparison_attention_n300_M6...,64000,1587200,246307,64,0.155183,2c465e03354342bb41455e6fbcfdda4e3e07c043cf757b...,5.246055,verified
3,4,v2_limited_sharing_control_attention_n300_M64000,64000,9603002,9154745,173,0.953321,1d8c93617f2806604694c75d2c1bee022cf6d0586cb919...,8.910215,verified
4,5,v2_main_comparison_attention_n496_M64000,59520,14677812,62468,264,0.004256,95bbb5275cad59f964561947b777065f6a224cf4f8cbd3...,7.919517,verified
5,6,v2_fixed_size_run_comparison_attention_n496_M6...,64000,1587200,246208,64,0.155121,1f854c8f5390551002dae75167e71f7981e4a95ecd06be...,8.142472,verified
6,7,v2_limited_sharing_control_attention_n496_M64000,64000,15877122,15430401,286,0.971864,85a2bd747a0e034b32643015043e7bc50b27def4d4f396...,15.300050,verified
7,8,v2_main_comparison_step_n300_M64000,63000,9431510,66105,173,0.007009,9a7d82c7847899ea9bb0c19810f98e463f4cf2690f11e6...,85.070204,verified
8,9,v2_main_comparison_step_n496_M64000,59520,14778502,62470,260,0.004227,7959adb2034a44912ea8abb662aa545e09c907c51dfde9...,85.365689,verified
9,10,v2_small_record_boundary_attention_n16_M65535,65535,524288,65535,16,0.124998,7680106e1ab4bb834380305677f2e74f52c966ff6180ab...,0.738225,verified


## Cell 4. Harness

Definitions only. Cells 1, 2 and 4 run on both machines. Cell 2 in verify mode supplies `input_path`, `generate_subsets`, `flatten_subsets`, `content_sha256` and `sha256_array` to this cell.

Loading an input checks the file SHA256 against the manifest, the content SHA256 against the stored arrays, and for attention rows regenerates W with `make_weights` and checks it against the stored SHA256. Preparation functions time the three components of protocol step 1. Shared preparation regenerates the subsets and, for attention rows, Q, K, V and W from the seed, and on its first repeat compares the regenerated content with the file's hash. Cache own preparation is the trie, the level plan and its transfer to the GPU, synchronized. Masked attention own preparation is the P × n boolean mask on the CPU.

Execution functions call the library as written. The cache runs `attn_cache` once, unchunked. Masked attention runs `sdpa_chunk` per chunk with the chunk's mask rows transferred to the GPU inside the call. Gathered attention runs `subset_attention_uncached` per chunk. The training step reproduces V2C4's `upstream`, `fwd_bwd` and `train_step`. The model, optimizer and random number generator states are restored from the file before every call, outside every clock.

Each evaluation runs, in order, the agreement call, the capture call, two warmups and five timed repeats. The agreement call is the method exactly as a timed call runs it, the complete training step with its update on training step rows, nothing collected, and its wall clock alone decides the ceiling. A nonreference method above the ceiling stops there, with status ceiling exceeded and an agreement row of status not run. Otherwise the capture call runs the method again, untimed, each output chunk copied into a preallocated host buffer as it completes, the forward and backward pass without the update on training step rows, and its tensors are compared with the reference under the float32 rules, or checked finite on the reference row. The reference runs its capture whatever its wall clock. A capture failure ends the evaluation with that status, stage capture. Every nonreference evaluation that runs writes exactly one `agreement_checks` row. `execution_calls` holds warmups and timed repeats only. Out of memory, unsupported and error are caught per call, cleanup included, and end the evaluation at that stage.

In [9]:
# Estimated run time: under 5 seconds on either machine. Definitions only.
import gc, copy
import torch.nn.functional as F
from set_attention_bench import subset_mask, attn_cache
from set_attention_data import batch_ret_loss
from set_attention_model import (SetModelConfig, SemanticMaskedSetEncoder, SubsetInteractionStack,
                                 encoder_layer_kv)

TOL32_OUT, TOL32_GRAD, TOL32_ZERO = (1e-6, 1e-4), 1e-3, 1e-6
TOL_LOSS, TOL_GRAD, TOL_ZERO = (1e-6, 1e-4), 1e-3, 1e-6
COMPARE_CHUNK = 65_536
CACHE, REPLAY = "cache", "cache replay"                 # eager level walk; the same call captured once in a CUDA graph and replayed
DIRECT_REPLAY = "masked attention replay"               # a masked attention combination captured once in a CUDA graph and replayed
CACHE_METHODS = (CACHE, REPLAY)
REPLAY_METHODS = (REPLAY, DIRECT_REPLAY)
GRAPH_WARMUPS = 3
DIRECT_REPLAY_MENU = [(DIRECT_REPLAY, b, c) for m, b, c in DIRECT_MENU if m == "masked attention"]   # gathered attention builds its indices on the host inside the call, so it has no replay
METHODS = [(CACHE, None, None), (REPLAY, None, None)] + DIRECT_MENU + DIRECT_REPLAY_MENU
EXEC_ORDER = [0]

def base_method(method):
    """The eager method a replay captures."""
    return {REPLAY: CACHE, DIRECT_REPLAY: "masked attention"}.get(method, method)

def method_label(method, backend, chunk):
    if method in CACHE_METHODS:
        return method
    if method == DIRECT_REPLAY:
        return combination_label("masked attention", backend, chunk) + " replay"
    return combination_label(method, backend, chunk)

def chunk_value(chunk):
    return None if chunk in (None, "all") else int(chunk)

def sync():
    torch.cuda.synchronize()

def clean():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

def mib(x):
    return x / 2 ** 20

def timed(fn):
    sync()
    t = time.perf_counter()
    out = fn()
    sync()
    return out, 1000 * (time.perf_counter() - t)

def classify(exc):
    """Status and detail for an exception raised inside a call."""
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return "out of memory", str(exc).splitlines()[0][:300]
    msg = str(exc)
    if isinstance(exc, NotImplementedError) or any(s in msg for s in ("No available kernel", "not implemented", "not supported", "Unsupported")):
        return "unsupported", msg[:300]
    return "error", (type(exc).__name__ + ": " + msg)[:300]

def tree_cpu(o):
    if torch.is_tensor(o):
        return o.detach().cpu().clone()
    if isinstance(o, dict):
        return {k: tree_cpu(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return type(o)(tree_cpu(v) for v in o)
    return copy.deepcopy(o)

def hash_tree(h, o):
    if torch.is_tensor(o):
        a = o.detach().cpu().contiguous().numpy() if o.dtype != torch.bfloat16 else o.detach().cpu().float().numpy()
        h.update(f"tensor:{a.dtype}:{a.shape}".encode()); h.update(memoryview(np.ascontiguousarray(a)).cast("B"))
    elif isinstance(o, np.ndarray):
        h.update(f"array:{o.dtype}:{o.shape}".encode()); h.update(memoryview(np.ascontiguousarray(o)).cast("B"))
    elif isinstance(o, dict):
        for k in sorted(o, key=str):
            h.update(f"key:{k}".encode()); hash_tree(h, o[k])
    elif isinstance(o, (list, tuple)):
        h.update(f"seq:{len(o)}".encode())
        for v in o:
            hash_tree(h, v)
    else:
        h.update(f"val:{o!r}".encode())

def subsets_sha256(indptr, flat):
    h = hashlib.sha256()
    hash_tree(h, [np.ascontiguousarray(indptr, dtype=np.int64), np.ascontiguousarray(flat, dtype=np.int16)])
    return h.hexdigest()

def step_content_sha256(indptr, flat, record, snap):
    h = hashlib.sha256()
    hash_tree(h, [np.ascontiguousarray(indptr, dtype=np.int64), np.ascontiguousarray(flat, dtype=np.int16),
                  record["cols"], record["vals"], record["name_vectors"],
                  snap["enc"], snap["stack"], snap["opt"], snap["rng_cpu"], snap["rng_cuda"]])
    return h.hexdigest()

def tuples_from_flat(indptr, flat):
    parts = np.split(flat.astype(np.int64), indptr[1:-1])
    return [tuple(p.tolist()) for p in parts]

# ----- loading -----
def load_input(row):
    path = input_path(row.benchmark_input_id)
    assert sha256_file(path) == row.benchmark_input_sha256, f"file hash differs from the manifest: {path}"
    rec = load_from_pickle(path)
    assert rec["benchmark_input_id"] == row.benchmark_input_id
    step = row.benchmark_type == "training step benchmark"
    if step:
        assert "subsets_sha256" in rec, f"training step file without subsets_sha256: {path}"
        got = step_content_sha256(rec["subset_indptr"], rec["subset_flat"], rec["record"], rec["state"])
    else:
        got = content_sha256(rec["subset_indptr"], rec["subset_flat"], rec["q"], rec["K"], rec["V"], rec["W_sha256"])
    assert got == rec["content_sha256"], f"content hash mismatch: {path}"
    ctx = {"row": row, "rec": rec, "step": step, "n": int(rec["n"]), "P": int(rec["P"]),
           "subs": tuples_from_flat(rec["subset_indptr"], rec["subset_flat"])}
    assert len(ctx["subs"]) == ctx["P"]
    ctx["intervals"] = run_intervals(rec)                  # rank intervals on run sampler and full power set rows, else None
    ctx["generated"] = ctx["intervals"] is not None        # the cache's trie is generated from the descriptor, no subset list
    if not step:
        W = make_weights(ctx["P"], C, D_K, rec["W_seed"], torch.float32, "cpu")
        assert sha256_array(W.numpy()) == rec["W_sha256"], "regenerated W differs from the stored hash"
        ctx["W_cpu"] = W
    return ctx

# ----- preparation, protocol step 1 -----
from set_attention_cache import powerset_run_offset, powerset_run_intervals, generate_run_trie

def run_intervals(rec):
    """Nonwrapping rank intervals of the row's family on run sampler and full power set rows, from the descriptor
    (n, M, run length, sampler key) through the sampler's own offset helper. None for the other samplers."""
    n, M, sel = int(rec["n"]), int(rec["budget_M"]), rec["subset_selection"]
    if sel == "run sampler":
        offset = powerset_run_offset(n, rec["sampler_key"], seed=SEED, epoch=0)
        return powerset_run_intervals(n, M, int(rec["requested_run_length_lambda"]), offset)
    if sel == "full power set":
        return [(0, powerset_total(n))]
    return None

def charged_to(ctx, comp):
    """The methods a preparation component is charged to. Each method pays for the preparation it needs."""
    if comp == "input generation":
        return [] if ctx["step"] else ["cache", "masked attention", "gathered attention"]
    if comp == "subset materialization":
        return ["masked attention", "gathered attention"] + ([] if ctx["generated"] else ["cache"])
    if comp == "cache own preparation":
        return ["cache"]
    return ["masked attention"]

def input_generation(ctx, repeat_index):
    """Q, K, V and W from the seed on attention rows, timed. Nothing on training step rows, whose model and state come
    from the file. First repeat: the regenerated tensors with the stored subset arrays checked against the file's hash."""
    rec = ctx["rec"]
    if ctx["step"]:
        return 0.0
    t0 = time.perf_counter()
    q, K, V = [t.detach() for t in make_inputs(ctx["n"], C, D_K, SEED, torch.float32, "cpu")]
    W = make_weights(ctx["P"], C, D_K, rec["W_seed"], torch.float32, "cpu")
    ms = 1000 * (time.perf_counter() - t0)
    if repeat_index == 1:
        got = content_sha256(rec["subset_indptr"], rec["subset_flat"], q, K, V, sha256_array(W.numpy()))
        assert got == rec["content_sha256"], "regenerated input differs from the file"
    return ms

def subset_materialization(ctx, repeat_index):
    """The subset list regenerated from the key, timed. First repeat: regenerated subsets checked against the file."""
    row, rec = ctx["row"], ctx["rec"]
    t0 = time.perf_counter()
    subsets = generate_subsets(row)
    indptr, flat = flatten_subsets(subsets)
    ms = 1000 * (time.perf_counter() - t0)
    if repeat_index == 1:
        assert subsets_sha256(indptr, flat) == subsets_sha256(rec["subset_indptr"], rec["subset_flat"]), \
            "regenerated subsets differ from the file"
    return ms

def cache_trie(ctx):
    """The cache's trie. Generated from the rank intervals on run sampler and full power set rows, the intervals recomputed
    from the descriptor inside the clock. Built from the subset list on the other rows. The same arrays either way (Cell 6)."""
    if ctx["generated"]:
        return generate_run_trie(ctx["n"], run_intervals(ctx["rec"]))
    return build_prefix_trie_canonical(ctx["subs"])

def cache_preparation(ctx):
    """Trie, level plan and its transfer to the GPU, synchronized. Returns (plan, ms)."""
    def build():
        return plan_to_device(forest_level_plan_np([cache_trie(ctx)], [0]), DEVICE)
    return timed(build)

def split_intervals(intervals, a, b):
    """The intervals of positions [a, b) of the sorted family, cut from its interval list."""
    out, pos = [], 0
    for start, length in intervals:
        lo, hi = max(a, pos), min(b, pos + length)
        if lo < hi:
            out.append((start + lo - pos, hi - lo))
        pos += length
    return out

def masked_preparation(ctx):
    """The P x n boolean mask on the CPU. Returns (mask, ms)."""
    t0 = time.perf_counter()
    m = subset_mask(ctx["subs"], ctx["n"], "cpu")
    return m, 1000 * (time.perf_counter() - t0)

# ----- attention benchmark execution -----
def attention_inputs(ctx):
    rec = ctx["rec"]
    q, K, V = [rec[k].to(DEVICE).requires_grad_() for k in ("q", "K", "V")]
    return {"q": q, "K": K, "V": V, "W": ctx["W_cpu"].to(DEVICE)}

def attention_call(g, ctx, method, backend, chunk, plan, mask_cpu, out_host=None, keep=None):
    """One execution of an attention benchmark. Gradients land in q, K, V. With out_host, a preallocated
    [P, C, d] host tensor, each output chunk is copied into it as it completes. With keep, a list, each output
    tensor stays referenced in it (graph capture, so replayed outputs are read without a copy). Masked attention
    moves each chunk's mask rows to the GPU inside the call; a mask already on the GPU (replay) is used in place."""
    q, K, V, W = g["q"], g["K"], g["V"], g["W"]
    for t in (q, K, V):
        t.grad = None
    P = ctx["P"]
    if method == "cache":
        if isinstance(plan, list):                       # run aligned groups, used only after the unchunked call ran out of memory
            ups = [scores(q, K), V]
            leaves = [u.detach().requires_grad_() for u in ups]
            for a, b, gplan in plan:
                o = subset_attention_levels_exact_bwd(leaves[0], leaves[1], gplan)
                (W[a:b] * o).sum().backward()
                if out_host is not None:
                    out_host[a:b].copy_(o.detach())
                del o
            torch.autograd.backward(ups, [lf.grad for lf in leaves])
            return None
        out = attn_cache(q, K, V, plan)
        (W * out).sum().backward()
        if out_host is not None:
            out_host.copy_(out.detach())
        if keep is not None:
            keep.append(out.detach())
        return None
    if method == "gathered attention":
        ups = [scores(q, K), V]
    else:
        ups = [q, *sdpa_layout(q, K, V)]
    leaves = [u.detach().requires_grad_() for u in ups]
    for sl in chunks_of(P, chunk_value(chunk)):
        if method == "gathered attention":
            o = subset_attention_uncached(leaves[0], leaves[1], ctx["subs"][sl])
        else:
            m = mask_cpu[sl].to(DEVICE)
            o = sdpa_chunk(leaves[0], leaves[1], leaves[2], m, backend == "math")
            del m
        (W[sl] * o).sum().backward()
        if out_host is not None:
            out_host[sl].copy_(o.detach())
        if keep is not None:
            keep.append(o.detach())
        del o
    torch.autograd.backward(ups, [lf.grad for lf in leaves])
    return None

def attention_result(g, out_host):
    return {"out": out_host, "grad_q": g["q"].grad.detach().cpu(), "grad_K": g["K"].grad.detach().cpu(),
            "grad_V": g["V"].grad.detach().cpu()}

def max_abs_chunked(a, b):
    m = 0.0
    for i in range(0, a.shape[0], COMPARE_CHUNK):
        m = max(m, float((a[i:i + COMPARE_CHUNK].double() - b[i:i + COMPARE_CHUNK].double()).abs().max()))
    return m

def max_abs_value_chunked(b):
    m = 0.0
    for i in range(0, b.shape[0], COMPARE_CHUNK):
        m = max(m, float(b[i:i + COMPARE_CHUNK].double().abs().max()))
    return m

def all_finite_chunked(a):
    return all(bool(torch.isfinite(a[i:i + COMPARE_CHUNK]).all()) for i in range(0, a.shape[0], COMPARE_CHUNK))

def attention_finite(res):
    return all(all_finite_chunked(res[k]) for k in ("out", "grad_q", "grad_K", "grad_V"))

def compare_attention(res, ref):
    """The float32 rules. Output: max absolute error <= 1e-6 + 1e-4 x max|reference|. Each gradient: relative
    Euclidean error <= 1e-3, or absolute <= 1e-6 when the reference gradient is zero."""
    if not attention_finite(res):
        return {"finite": False, "output_max_abs_error": None, "gradient_max_relative_error": None,
                "tensor_checks": {}, "status": "failed agreement", "detail": "nonfinite output or gradient"}
    checks, rel_errors = {}, []
    mad, mref = max_abs_chunked(res["out"], ref["out"]), max_abs_value_chunked(ref["out"])
    tol = TOL32_OUT[0] + TOL32_OUT[1] * mref
    checks["out"] = {"rule": "absolute", "error": mad, "tolerance": tol, "pass": bool(mad <= tol)}
    for k in ("grad_q", "grad_K", "grad_V"):
        a, b = res[k].double(), ref[k].double()
        nd, nr = float((a - b).norm()), float(b.norm())
        if nr > 0:
            checks[k] = {"rule": "relative", "error": nd / nr, "tolerance": TOL32_GRAD, "pass": bool(nd / nr <= TOL32_GRAD)}
            rel_errors.append(nd / nr)
        else:
            checks[k] = {"rule": "absolute", "error": nd, "tolerance": TOL32_ZERO, "pass": bool(nd <= TOL32_ZERO)}
    failed = [k for k, c in checks.items() if not c["pass"]]
    return {"finite": True, "output_max_abs_error": mad,
            "gradient_max_relative_error": (max(rel_errors) if rel_errors else None), "tensor_checks": checks,
            "status": "failed agreement" if failed else "passed",
            "detail": (None if not failed else "; ".join(f"{k} error {checks[k]['error']:.3e} tolerance {checks[k]['tolerance']:.3e}" for k in failed))}

# ----- training step execution, V2C4's step -----
def step_models(ctx):
    rec, snap = ctx["rec"], ctx["rec"]["state"]
    cfg = SetModelConfig(**rec["model_config"])
    enc = SemanticMaskedSetEncoder(cfg)
    enc.load_state_dict(snap["enc"])
    stack = SubsetInteractionStack(cfg.n_layers, cfg.n_heads, cfg.dim // cfg.n_heads, rec["n_queries"], rec["hidden"])
    stack.load_state_dict(snap["stack"])
    enc, stack = enc.to(DEVICE, torch.float32), stack.to(DEVICE, torch.float32)
    enc.train(); stack.train()
    opt = torch.optim.AdamW([{"params": enc.parameters(), "lr": rec["lr"] * rec["enc_scale"]},
                             {"params": stack.parameters(), "lr": rec["lr"]}], weight_decay=rec["weight_decay"])
    opt.load_state_dict(tree_cpu(snap["opt"]))
    r = rec["record"]
    ft = {"nv": torch.from_numpy(r["name_vectors"]).to(DEVICE), "vals": torch.from_numpy(r["vals"]).to(DEVICE),
          "pad": torch.zeros((1, r["n"]), dtype=torch.bool, device=DEVICE),
          "y": torch.full((1,), float(rec["y_target"]), dtype=torch.float32, device=DEVICE)}
    params = list(enc.parameters()) + list(stack.parameters())
    names_p = [f"enc.{nm}" for nm, _ in enc.named_parameters()] + [f"stack.{nm}" for nm, _ in stack.named_parameters()]
    n_enc = sum(p.numel() for p in enc.parameters())
    s = {"enc": enc, "stack": stack, "opt": opt, "ft": ft, "snap": snap, "params": params, "names_p": names_p,
         "groups": {"all": slice(0, None), "encoder": slice(0, n_enc), "stack": slice(n_enc, None)},
         "n_queries": rec["n_queries"],
         "counts_t": torch.tensor([ctx["P"]], dtype=torch.long, device=DEVICE)}   # batch_ret_loss counts, built once on the GPU
    with torch.no_grad():
        upstream(s)                       # one pass with the encoder's input checks on, over the benchmark's fixed record
    enc.check_inputs = False              # every later call, for every method, skips those per call host reads
    return s

def restore(s):
    s["enc"].load_state_dict(s["snap"]["enc"])
    s["stack"].load_state_dict(s["snap"]["stack"])
    s["opt"].load_state_dict(tree_cpu(s["snap"]["opt"]))
    s["opt"].zero_grad(set_to_none=True)
    torch.set_rng_state(s["snap"]["rng_cpu"])
    torch.cuda.set_rng_state(s["snap"]["rng_cuda"])

def upstream(s):
    enc, stack, ft = s["enc"], s["stack"], s["ft"]
    kv = encoder_layer_kv(enc, ft["nv"][None], ft["vals"][None], ft["pad"])
    z, V = stack.scores_values([(K[0], Vv[0]) for K, Vv in kv])
    n, Cc, dh = V.shape
    w = torch.softmax(z.view(1, n, Cc).masked_fill(ft["pad"][..., None], float("-inf")), dim=1)
    pf = stack.predict(torch.einsum("blc,blcd->bcd", w, V.view(1, n, Cc, dh)))
    return kv, z, V, pf

def channel_qkv(s, kv, V):
    n, _, dh = V.shape
    Kc = torch.cat([K[0].unsqueeze(2).expand(-1, -1, s["n_queries"], -1).reshape(n, -1, dh) for K, _ in kv], 1)
    return s["stack"].q.reshape(-1, dh), Kc

def fwd_bwd(s, ctx, method, backend, chunk, plan, mask_cpu):
    """Forward and backward of one complete step's loss, as V2C4. Returns the detached loss."""
    stack, ft, P = s["stack"], s["ft"], ctx["P"]
    kv, z, V, pf = upstream(s)
    y = ft["y"]
    if method == "cache":
        if isinstance(plan, list):                       # run aligned groups: per subset terms, global normalization by P, as the direct path
            l_full = F.huber_loss(pf, y, delta=1.0)
            ups = [z, V]
            leaves = [u.detach().requires_grad_() for u in ups]
            total = 0.5 * l_full.detach()
            for a, b, gplan in plan:
                o = subset_attention_levels_exact_bwd(leaves[0], leaves[1], gplan)
                ps = stack.predict(o)
                term = 0.5 * F.huber_loss(ps, y.expand_as(ps), delta=1.0, reduction="sum") / P
                term.backward()
                total = total + term.detach()
                del o, ps, term
            torch.autograd.backward([0.5 * l_full] + ups, [None] + [lf.grad for lf in leaves])
            return total
        ps = stack.predict(subset_attention_levels_exact_bwd(z, V, plan))
        loss = batch_ret_loss(pf, ps, s["counts_t"], y)
        loss.backward()
        return loss.detach()
    l_full = F.huber_loss(pf, y, delta=1.0)
    if method == "gathered attention":
        ups = [z, V]
    else:
        qC, Kc = channel_qkv(s, kv, V)
        ups = [qC, *sdpa_layout(qC, Kc, V)]
    leaves = [u.detach().requires_grad_() for u in ups]
    total = 0.5 * l_full.detach()
    for sl in chunks_of(P, chunk_value(chunk)):
        if method == "gathered attention":
            o = subset_attention_uncached(leaves[0], leaves[1], ctx["subs"][sl])
        else:
            m = mask_cpu[sl].to(DEVICE)
            o = sdpa_chunk(leaves[0], leaves[1], leaves[2], m, backend == "math")
            del m
        ps = stack.predict(o)
        term = 0.5 * F.huber_loss(ps, y.expand_as(ps), delta=1.0, reduction="sum") / P
        term.backward()
        total = total + term.detach()
        del o, ps, term
    torch.autograd.backward([0.5 * l_full] + ups, [None] + [lf.grad for lf in leaves])
    return total

def train_step(s, ctx, method, backend, chunk, plan, mask_cpu):
    s["opt"].zero_grad(set_to_none=True)
    loss = fwd_bwd(s, ctx, method, backend, chunk, plan, mask_cpu)
    s["opt"].step()
    return loss

def grad_vec(s):
    used = np.array([p.grad is not None for p in s["params"]])
    g = torch.cat([(torch.zeros_like(p) if p.grad is None else p.grad).reshape(-1).double().cpu() for p in s["params"]])
    return g, used

def step_result(s, loss):
    g, used = grad_vec(s)
    return {"loss": float(loss), "grad": g, "used": used}

def step_finite(res):
    return bool(np.isfinite(res["loss"])) and bool(torch.isfinite(res["grad"]).all())

def compare_step(res, ref, s):
    """V2C4's rules. Same parameters receive gradients. Loss within 1e-6 + 1e-4 x |reference|. Gradient relative
    Euclidean error <= 1e-3 for all parameters, the encoder and the stack, absolute 1e-6 against a zero reference."""
    finite = step_finite(res)
    same = bool(np.array_equal(res["used"], ref["used"]))
    out = {"finite": finite, "same_gradient_parameters": same, "loss_absolute_difference": None,
           "loss_tolerance": None, "parameter_gradient_relative_difference": None,
           "parameter_gradient_tolerance": TOL_GRAD, "tensor_checks": {}}
    if not finite:
        out.update(status="failed agreement", detail="nonfinite loss or gradient")
        return out
    ld = abs(res["loss"] - ref["loss"])
    lt = TOL_LOSS[0] + TOL_LOSS[1] * abs(ref["loss"])
    out.update(loss_absolute_difference=ld, loss_tolerance=lt)
    checks = {"loss": {"rule": "absolute", "error": ld, "tolerance": lt, "pass": bool(ld <= lt)}}
    for gname, sl in s["groups"].items():
        a, b = res["grad"][sl], ref["grad"][sl]
        nd, nr = float((a - b).norm()), float(b.norm())
        if nr > 0:
            checks[f"grad_{gname}"] = {"rule": "relative", "error": nd / nr, "tolerance": TOL_GRAD, "pass": bool(nd / nr <= TOL_GRAD)}
        else:
            checks[f"grad_{gname}"] = {"rule": "absolute", "error": nd, "tolerance": TOL_ZERO, "pass": bool(nd <= TOL_ZERO)}
    out["parameter_gradient_relative_difference"] = checks["grad_all"]["error"]
    failed = [k for k, c in checks.items() if not c["pass"]]
    if not same:
        bad = [s["names_p"][i] for i in np.flatnonzero(res["used"] != ref["used"])][:10]
        checks["gradient_presence"] = {"rule": "equality", "error": len(bad), "tolerance": 0, "pass": False, "parameters": bad}
        failed.append("gradient_presence")
    out["tensor_checks"] = checks
    out["status"] = "failed agreement" if failed else "passed"
    out["detail"] = None if not failed else "; ".join(failed)
    return out

# ----- rows -----
def new_evaluation_row(row, method, backend, chunk):
    label = method_label(method, backend, chunk)
    return {"evaluation_id": f"{row.benchmark_id}#{label}", "benchmark_id": row.benchmark_id, "method": method,
            "attention_backend": backend, "chunk_size": chunk_value(chunk), "chunked": chunk_value(chunk) is not None,
            "direct_combination": None if method in CACHE_METHODS else label,
            "traversal": "level walk" if method in CACHE_METHODS else None,
            "backward": "closed form backward pass" if method in CACHE_METHODS else "autograd",
            "execution_mode": "graph replay" if method in REPLAY_METHODS else "eager",
            "implementation_commit": IMPLEMENTATION_COMMIT, "implementation_file_sha256": IMPLEMENTATION_FILE_SHA256,
            "is_reference": (method, backend, chunk) == REFERENCE_COMBINATION, "reference_finite": None,
            "status": "not run budget exhausted", "status_stage": "not started", "detail": None,
            "agreement_call_wall_ms": None, "warmups_completed": 0, "timed_repeats_completed": 0,
            "execution_median_ms": None, "execution_min_ms": None, "execution_max_ms": None,
            "peak_gpu_allocated_median_mib": None, "peak_gpu_reserved_median_mib": None,
            "cache_groups": None, "cache_group_cut_shared_prefix": None,
            "cache_group_preparation_ms": None, "cache_group_attempts": None,
            "replay_setup_ms": None, "replay_capture_ms": None, "replay_reserved_after_capture_mib": None,
            "replay_mask_transfer_ms": None, "replay_verification": None}

def new_agreement_row(ev, ref_id, status, wall=None, exceeded=False, detail=None, cmp=None):
    cmp = cmp or {}
    return {"evaluation_id": ev["evaluation_id"], "reference_evaluation_id": ref_id, "status": status,
            "finite": cmp.get("finite"), "output_max_abs_error": cmp.get("output_max_abs_error"),
            "gradient_max_relative_error": cmp.get("gradient_max_relative_error"),
            "tensor_checks": cmp.get("tensor_checks", {}),
            "loss_absolute_difference": cmp.get("loss_absolute_difference"), "loss_tolerance": cmp.get("loss_tolerance"),
            "parameter_gradient_relative_difference": cmp.get("parameter_gradient_relative_difference"),
            "parameter_gradient_tolerance": cmp.get("parameter_gradient_tolerance"),
            "same_gradient_parameters": cmp.get("same_gradient_parameters"),
            "wall_ms": wall, "ceiling_exceeded": bool(exceeded), "detail": detail}

def one_call(ev, call_type, call_index, fn, calls, step, record):
    """Cleanup, then fn under the memory and wall clock capture, every failure caught. Appends an
    execution_calls row when record. Returns (status, result, wall_ms, detail)."""
    rowc = {"evaluation_id": ev["evaluation_id"], "call_type": call_type, "call_index": call_index,
            "execution_order": None, "status": "completed", "wall_ms": None,
            "gpu_allocated_start_mib": None, "peak_gpu_allocated_mib": None, "peak_gpu_reserved_mib": None,
            "loss": None, "detail": None}
    if record:
        EXEC_ORDER[0] += 1
        rowc["execution_order"] = EXEC_ORDER[0]
    try:
        clean()
        rowc["gpu_allocated_start_mib"] = mib(torch.cuda.memory_allocated())
        result, ms = timed(fn)
        rowc["wall_ms"], rowc["peak_gpu_allocated_mib"] = ms, mib(torch.cuda.max_memory_allocated())
        rowc["peak_gpu_reserved_mib"] = mib(torch.cuda.max_memory_reserved())
        if step and record and result is not None:
            rowc["loss"] = float(result)
        if record:
            calls.append(rowc)
        return "completed", result, ms, None
    except Exception as exc:
        rowc["status"], rowc["detail"] = classify(exc)
        try:
            rowc["peak_gpu_allocated_mib"] = mib(torch.cuda.max_memory_allocated())
        except Exception:
            pass
        if record:
            calls.append(rowc)
        return rowc["status"], None, None, rowc["detail"]

def evaluate_method(ctx, row, method, backend, chunk, plan, mask_cpu, ref, s, calls, agreements, gs=None):
    """Agreement call, ceiling, capture call, comparison, warmups, timed repeats. Appends exactly one
    agreement_checks row for a nonreference method. Returns (evaluation row, reference result or None).
    With gs, a captured graph from build_replay, its own timed, capture and prepare functions replace the eager ones."""
    ev = new_evaluation_row(row, method, backend, chunk)
    step, is_ref = ctx["step"], ev["is_reference"]
    ref_id = None if ref is None else ref["evaluation_id"]
    ev["status_stage"] = "reference" if is_ref else "agreement"

    def fail(status, detail, wall=None, exceeded=False):
        if exceeded:
            detail = f"{detail}; agreement call {wall:.0f} ms above the ceiling"
        ev["status"], ev["detail"] = status, detail
        if not is_ref:
            agreements.append(new_agreement_row(ev, ref_id, status, wall, exceeded, detail))
        return ev, None

    g = None
    if not step and gs is None:
        try:
            g = attention_inputs(ctx)
        except Exception as exc:
            return fail(*classify(exc))

    def timed_fn():
        if gs is not None:
            return gs["timed"]()
        if step:
            return train_step(s, ctx, method, backend, chunk, plan, mask_cpu)
        return attention_call(g, ctx, method, backend, chunk, plan, mask_cpu)

    def capture_fn():
        if gs is not None:
            return gs["capture"]()
        if step:
            return step_result(s, fwd_bwd(s, ctx, method, backend, chunk, plan, mask_cpu))
        out_host = torch.empty((ctx["P"], C, D_K), dtype=torch.float32)
        attention_call(g, ctx, method, backend, chunk, plan, mask_cpu, out_host=out_host)
        return attention_result(g, out_host)

    def prepare_state():
        if gs is not None:
            gs["prepare"]()
        elif step:
            restore(s)

    # agreement call: the method as timed, nothing collected; its wall clock decides the ceiling
    try:
        prepare_state()
    except Exception as exc:
        return fail(*classify(exc))
    status, _, wall, detail = one_call(ev, ev["status_stage"], 1, timed_fn, calls, step, record=False)
    ev["agreement_call_wall_ms"] = wall
    if status != "completed":
        return fail(status, detail)
    exceeded = bool(wall > CEILING_MS)
    if exceeded and not is_ref:
        ev["status"], ev["detail"] = "ceiling exceeded", f"agreement call {wall:.0f} ms above {CEILING_MS:.0f} ms"
        agreements.append(new_agreement_row(ev, ref_id, "not run", wall, True, ev["detail"]))
        return ev, None

    # capture call: untimed, outputs into a preallocated host buffer
    ev["status_stage"] = "capture"
    try:
        prepare_state()
    except Exception as exc:
        return fail(*classify(exc), wall, exceeded)
    status, res, _, detail = one_call(ev, "capture", 1, capture_fn, calls, step, record=False)
    if status != "completed":
        return fail(status, detail, wall, exceeded)

    try:
        if is_ref:
            finite = step_finite(res) if step else attention_finite(res)
            ev["reference_finite"] = bool(finite)
            if not finite:
                ev["status"], ev["detail"] = "reference unavailable", "reference output or gradient not finite"
                return ev, None
            ref_out = dict(res, evaluation_id=ev["evaluation_id"])
        else:
            cmp = compare_step(res, ref, s) if step else compare_attention(res, ref)
            agreements.append(new_agreement_row(ev, ref_id, cmp["status"], wall, exceeded, cmp["detail"], cmp))
            if cmp["status"] != "passed":
                ev["status"], ev["detail"] = "failed agreement", cmp["detail"]
                return ev, None
            ref_out = None
    except Exception as exc:
        return fail(*classify(exc), wall, exceeded)
    finally:
        del res
    if exceeded:
        ev["status"], ev["detail"] = "ceiling exceeded", f"agreement call {wall:.0f} ms above {CEILING_MS:.0f} ms"
        return ev, ref_out

    def run_calls(call_type, count, counter_key):
        for i in range(1, count + 1):
            try:
                prepare_state()
            except Exception as exc:
                ev["status"], ev["detail"] = classify(exc)
                return False
            status, _, _, detail = one_call(ev, call_type, i, timed_fn, calls, step, record=True)
            if status != "completed":
                ev["status"], ev["detail"] = status, detail
                return False
            ev[counter_key] = i
        return True

    ev["status_stage"] = "warmup"
    if not run_calls("warmup", WARMUPS, "warmups_completed"):
        return ev, ref_out
    ev["status_stage"] = "timed repeat"
    if not run_calls("timed repeat", TIMED_REPEATS, "timed_repeats_completed"):
        return ev, ref_out
    mine = [c for c in calls if c["evaluation_id"] == ev["evaluation_id"] and c["call_type"] == "timed repeat"]
    walls = [c["wall_ms"] for c in mine]
    peaks = [c["peak_gpu_allocated_mib"] for c in mine]
    peaks_r = [c["peak_gpu_reserved_mib"] for c in mine]
    ev.update(status="timed", execution_median_ms=float(np.median(walls)), execution_min_ms=float(min(walls)),
              execution_max_ms=float(max(walls)), peak_gpu_allocated_median_mib=float(np.median(peaks)),
              peak_gpu_reserved_median_mib=float(np.median(peaks_r)))
    return ev, ref_out

# ----- cache fallback to run aligned groups, only when the unchunked evaluation runs out of memory -----
CACHE_GROUPS_MAX = 8
CACHE_CUT_WINDOW = 0.01

def _lcp(x, y):
    k = 0
    for u, v in zip(x, y):
        if u != v:
            break
        k += 1
    return k

def grouped_cache_plans(ctx, g):
    """g contiguous groups of the sorted family. Each cut sits at the position with the fewest shared prefix features within
    CACHE_CUT_WINDOW of P of its equal split. One trie and level plan per group, on the GPU, synchronized.
    Returns ([(a, b, plan)], shared prefix length at each cut, ms)."""
    subs, P = ctx["subs"], ctx["P"]
    w = max(1, int(CACHE_CUT_WINDOW * P))
    def build():
        cuts, shared = [0], []
        for j in range(1, g):
            c = round(j * P / g)
            l, _, i = min((_lcp(subs[i - 1], subs[i]), abs(i - c), i) for i in range(max(1, c - w), min(P, c + w + 1)))
            cuts.append(i); shared.append(l)
        cuts.append(P)
        def group_trie(a, b):
            return (generate_run_trie(ctx["n"], split_intervals(ctx["intervals"], a, b)) if ctx["generated"]
                    else build_prefix_trie_canonical(subs[a:b]))
        groups = [(a, b, plan_to_device(forest_level_plan_np([group_trie(a, b)], [0]), DEVICE))
                  for a, b in zip(cuts[:-1], cuts[1:])]
        return groups, shared
    (groups, shared), ms = timed(build)
    return groups, shared, ms

def evaluate_cache(ctx, row, plan, mask_cpu, ref, s, calls, agreements):
    """The cache evaluation, unchunked first, exactly as evaluate_method runs it. Only if it ends out of memory, at any stage,
    its rows are removed and the whole evaluation runs again over g = 2, 3, ... run aligned groups, up to CACHE_GROUPS_MAX.
    The evaluation row records the groups used, the shared prefix at each cut, the group preparation time and the attempts."""
    ev, ref_out = evaluate_method(ctx, row, "cache", None, None, plan, mask_cpu, ref, s, calls, agreements)
    g, attempts, prep_ms, shared = 1, [], None, []
    while ev["status"] == "out of memory" and g < CACHE_GROUPS_MAX:
        eid = ev["evaluation_id"]
        attempts.append({"groups": g, "status_stage": ev["status_stage"], "detail": ev["detail"]})
        calls[:] = [c for c in calls if c["evaluation_id"] != eid]
        agreements[:] = [a for a in agreements if a["evaluation_id"] != eid]
        g += 1
        groups = None
        try:
            clean()
            groups, shared, prep_ms = grouped_cache_plans(ctx, g)
            ev, ref_out = evaluate_method(ctx, row, "cache", None, None, groups, mask_cpu, ref, s, calls, agreements)
        except Exception as exc:
            st, dt = classify(exc)
            ev = new_evaluation_row(row, "cache", None, None)
            ev.update(status=st, status_stage="agreement", detail=f"group plans: {dt}")
            agreements.append(new_agreement_row(ev, f"{row.benchmark_id}#{method_label(*REFERENCE_COMBINATION)}", st, None, False, ev["detail"]))
        finally:
            del groups
            clean()
    ev.update(cache_groups=g, cache_group_cut_shared_prefix=shared, cache_group_preparation_ms=prep_ms, cache_group_attempts=attempts)
    return ev, ref_out

# ----- replay verification, untimed, before any timed call -----
REPLAY_CHECKS = 3                                       # replays compared with each other; a training step replay is also compared with the eager step

def flat_params(s):
    return torch.cat([p.detach().reshape(-1).double().cpu() for p in s["params"]])

def flat_adam_state(opt, s):
    """exp_avg, exp_avg_sq and the step counters of every parameter, in parameter order."""
    m, v, steps = [], [], []
    for p in s["params"]:
        st = opt.state.get(p, {})
        if "exp_avg" in st:
            m.append(st["exp_avg"].detach().reshape(-1).double().cpu())
            v.append(st["exp_avg_sq"].detach().reshape(-1).double().cpu())
            steps.append(float(st["step"]))
    return (torch.cat(m) if m else torch.zeros(0, dtype=torch.float64)), (torch.cat(v) if v else torch.zeros(0, dtype=torch.float64)), np.array(steps)

def vector_check(a, b):
    """Relative Euclidean error <= TOL_GRAD, or absolute <= TOL_ZERO against a zero reference; nonfinite fails."""
    if not (bool(torch.isfinite(a).all()) and bool(torch.isfinite(b).all())) or a.shape != b.shape:
        return {"rule": "finite and shape", "error": None, "tolerance": None, "pass": False}
    nd, nr = float((a - b).norm()), float(b.norm())
    if nr > 0:
        return {"rule": "relative", "error": nd / nr, "tolerance": TOL_GRAD, "pass": bool(nd / nr <= TOL_GRAD)}
    return {"rule": "absolute", "error": nd, "tolerance": TOL_ZERO, "pass": bool(nd <= TOL_ZERO)}

def compare_update(res, ref):
    """A complete training step against another: loss (TOL_LOSS), gradients, the parameter update (post minus pre),
    both AdamW moment buffers (vector_check) and the step counters (equal)."""
    finite = bool(np.isfinite(res["loss"]))
    lt = TOL_LOSS[0] + TOL_LOSS[1] * abs(ref["loss"])
    ld = abs(res["loss"] - ref["loss"]) if finite else None
    checks = {"loss": {"rule": "absolute", "error": ld, "tolerance": lt, "pass": bool(finite and ld <= lt)},
              "gradients": vector_check(res["grad"], ref["grad"]),
              "same_gradient_parameters": {"rule": "equality", "error": None, "tolerance": None,
                                           "pass": bool(np.array_equal(res["used"], ref["used"]))},
              "parameter_update": vector_check(res["delta"], ref["delta"]),
              "exp_avg": vector_check(res["m"], ref["m"]),
              "exp_avg_sq": vector_check(res["v"], ref["v"]),
              "step_counters": {"rule": "equality", "error": None, "tolerance": None,
                                "pass": bool(np.array_equal(res["steps"], ref["steps"]))}}
    failed = [k for k, c in checks.items() if not c["pass"]]
    return {"status": "failed agreement" if failed else "passed", "detail": "; ".join(failed) if failed else None, "checks": checks}

def verification_summary(comparisons):
    """comparisons: {name: comparison dict with status, detail}. Passed only if every comparison passed."""
    failed = [f"{k}: {c['detail']}" for k, c in comparisons.items() if c["status"] != "passed"]
    return {"status": "failed agreement" if failed else "passed", "detail": " | ".join(failed) if failed else None,
            "comparisons": comparisons}

# ----- replay: the cache call, or a masked attention combination, captured once in a CUDA graph, then replayed -----
def build_attention_replay(ctx, method, backend, chunk, plan, mask_dev):
    """Captures forward and backward of one attention call (the cache, or a masked attention combination with its mask
    already on the GPU) on its own q, K, V, W. Each timed call copies the benchmark's values into the graph's input
    tensors and replays. The outputs stay referenced inside the graph. verify replays REPLAY_CHECKS times and compares
    every replay's output and gradients with the first, so restoration holds and gradients do not accumulate.
    Returns the replay functions and setup timings."""
    g = attention_inputs(ctx)
    q, K, V = g["q"], g["K"], g["V"]
    vals = tuple(t.detach().clone() for t in (q, K, V))
    bm = base_method(method)
    t_setup = time.perf_counter()
    side = torch.cuda.Stream()
    side.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side):
        for _ in range(GRAPH_WARMUPS):
            attention_call(g, ctx, bm, backend, chunk, plan, mask_dev)
    torch.cuda.current_stream().wait_stream(side)
    sync()
    for t in (q, K, V):
        t.grad = None
    graph, static_outs = torch.cuda.CUDAGraph(), []
    t_cap = time.perf_counter()
    with torch.cuda.graph(graph):
        attention_call(g, ctx, bm, backend, chunk, plan, mask_dev, keep=static_outs)
    sync()
    capture_ms, setup_ms = 1000 * (time.perf_counter() - t_cap), 1000 * (time.perf_counter() - t_setup)
    static_grads = (q.grad, K.grad, V.grad)

    def timed():
        with torch.no_grad():
            q.copy_(vals[0]); K.copy_(vals[1]); V.copy_(vals[2])
        graph.replay()
        return None

    def capture():
        timed()
        sync()
        out = torch.cat([o.cpu() for o in static_outs]) if len(static_outs) > 1 else static_outs[0].cpu()
        return {"out": out, "grad_q": static_grads[0].detach().cpu(),
                "grad_K": static_grads[1].detach().cpu(), "grad_V": static_grads[2].detach().cpu()}

    def verify():
        reps = [capture() for _ in range(REPLAY_CHECKS)]
        return verification_summary({f"replay {i} against replay 1": compare_attention(r, reps[0])
                                     for i, r in enumerate(reps[1:], 2)})

    return {"graph": graph, "timed": timed, "capture": capture, "verify": verify, "prepare": lambda: None,
            "release": lambda: None, "keep": [g, vals, static_outs, static_grads], "setup_ms": setup_ms, "capture_ms": capture_ms}

def build_step_replay(ctx, method, backend, chunk, plan, mask_dev, s):
    """Captures the complete training step, forward, backward and the AdamW update, with a capturable AdamW over
    the same parameters and the snapshot's state. Before every call the snapshot is written in place into the parameters,
    the optimizer state and the random number state, as restore does for the eager step. verify replays REPLAY_CHECKS
    times from the snapshot and compares loss, gradients, the parameter update, both AdamW moment buffers and the step
    counters of every replay with the first, then the first replay with the eager step of the same method from the
    same snapshot."""
    opt_g = torch.optim.AdamW([{"params": s["enc"].parameters()}, {"params": s["stack"].parameters()}])
    opt_g.load_state_dict(tree_cpu(s["snap"]["opt"]))
    for grp in opt_g.param_groups:
        grp["capturable"] = True
    for st in opt_g.state.values():
        if "step" in st and torch.is_tensor(st["step"]):
            st["step"] = st["step"].to(DEVICE)
    plist = [p for grp in opt_g.param_groups for p in grp["params"]]
    sg = dict(s, opt=opt_g)
    ptrs0 = [p.data_ptr() for p in s["params"]]
    bm = base_method(method)

    def restore_inplace():
        sg["enc"].load_state_dict(sg["snap"]["enc"])
        sg["stack"].load_state_dict(sg["snap"]["stack"])
        with torch.no_grad():
            for i, st in sg["snap"]["opt"]["state"].items():
                live = opt_g.state[plist[int(i)]]
                for k, v in st.items():
                    if torch.is_tensor(v):
                        live[k].copy_(v)
        torch.set_rng_state(sg["snap"]["rng_cpu"])
        torch.cuda.set_rng_state(sg["snap"]["rng_cuda"])

    t_setup = time.perf_counter()
    restore_inplace()
    side = torch.cuda.Stream()
    side.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side):
        for _ in range(GRAPH_WARMUPS):
            opt_g.zero_grad(set_to_none=True)
            fwd_bwd(sg, ctx, bm, backend, chunk, plan, mask_dev)
            opt_g.step()
    torch.cuda.current_stream().wait_stream(side)
    sync()
    restore_inplace()
    opt_g.zero_grad(set_to_none=True)
    graph = torch.cuda.CUDAGraph()
    t_cap = time.perf_counter()
    with torch.cuda.graph(graph):
        static_loss = fwd_bwd(sg, ctx, bm, backend, chunk, plan, mask_dev)
        opt_g.step()
    sync()
    capture_ms, setup_ms = 1000 * (time.perf_counter() - t_cap), 1000 * (time.perf_counter() - t_setup)
    assert [p.data_ptr() for p in s["params"]] == ptrs0, "replay: parameter tensors moved during capture"
    static_grads = [p.grad for p in s["params"]]                # the graph's gradient buffers

    def timed():
        graph.replay()
        return static_loss

    def capture():
        graph.replay()
        sync()
        return step_result(sg, static_loss)          # loss and gradients of the replayed step, taken before its update

    def after_step(opt, loss, pre):
        r = step_result(sg, loss)
        r["delta"] = flat_params(s) - pre
        r["m"], r["v"], r["steps"] = flat_adam_state(opt, s)
        return r

    def verify():
        reps = []
        for _ in range(REPLAY_CHECKS):
            restore_inplace()
            pre = flat_params(s)
            graph.replay()
            sync()
            reps.append(after_step(opt_g, static_loss, pre))
        restore(s)                                              # the eager optimizer and parameters from the same snapshot
        pre = flat_params(s)
        loss_e = train_step(s, ctx, bm, backend, chunk, plan, mask_dev)
        sync()
        eager = after_step(s["opt"], loss_e, pre)
        restore(s)                                              # the eager optimizer back to the snapshot (also clears gradients)
        for p, gref in zip(s["params"], static_grads):          # give the parameters back the graph's gradient buffers
            p.grad = gref
        restore_inplace()
        comps = {f"replay {i} against replay 1": compare_update(r, reps[0]) for i, r in enumerate(reps[1:], 2)}
        comps["replay 1 against the eager step"] = compare_update(reps[0], eager)
        return verification_summary(comps)

    def release():
        for p in s["params"]:
            p.grad = None

    return {"graph": graph, "timed": timed, "capture": capture, "verify": verify, "prepare": restore_inplace, "release": release,
            "keep": [opt_g, sg, static_loss, static_grads], "setup_ms": setup_ms, "capture_ms": capture_ms}

def evaluate_replay(ctx, row, method, backend, chunk, plan, mask_cpu, ref, s, calls, agreements):
    """A replay evaluation, cache replay or masked attention replay. For masked attention the whole mask is moved to
    the GPU once, timed and recorded as replay_mask_transfer_ms. Setup (graph warmups and capture) is timed once and
    recorded in the evaluation row. Then, untimed, verify checks repeated replays against each other and, for a
    training step, the replayed update against the eager step; a failure ends the evaluation as failed agreement with
    no timed call. Then the evaluation runs exactly as evaluate_method runs every other method. No run aligned group
    fallback: out of memory during capture or verification is the evaluation's result. The graph, its memory pool, the
    GPU mask and the cuBLAS workspaces created during capture are released before the next method."""
    ref_id = None if ref is None else ref["evaluation_id"]
    gs, mask_dev, transfer_ms, stage, reserved = None, None, None, "graph capture", None

    def release_all():
        nonlocal gs, mask_dev
        if gs is not None:
            gs["release"]()
        elif ctx["step"]:
            for p in s["params"]:
                p.grad = None
        gs, mask_dev = None, None
        clean()
        torch._C._cuda_clearCublasWorkspaces()   # cuBLAS caches a workspace per stream; those made during capture live in the graph's pool
        clean()

    def failed(status, detail, vr=None):
        ev = new_evaluation_row(row, method, backend, chunk)
        ev.update(status=status, status_stage=stage, detail=detail, replay_mask_transfer_ms=transfer_ms,
                  replay_reserved_after_capture_mib=reserved, replay_verification=vr)
        agreements.append(new_agreement_row(ev, ref_id, status, None, False, detail))
        return ev

    try:
        clean()
        if method == DIRECT_REPLAY:
            mask_dev, transfer_ms = timed(lambda: mask_cpu.to(DEVICE))
        r0 = torch.cuda.memory_reserved()
        gs = (build_step_replay(ctx, method, backend, chunk, plan, mask_dev, s) if ctx["step"]
              else build_attention_replay(ctx, method, backend, chunk, plan, mask_dev))
        reserved = mib(torch.cuda.memory_reserved() - r0)
        stage = "replay verification"
        vr = gs["verify"]()
    except Exception as exc:
        st, dt = classify(exc)
        ev = failed(st, f"{stage}: {dt}")
        release_all()
        return ev, None
    if vr["status"] != "passed":
        ev = failed("failed agreement", f"replay verification: {vr['detail']}", vr)
        release_all()
        return ev, None
    try:
        ev, _ = evaluate_method(ctx, row, method, backend, chunk, plan, mask_dev, ref, s, calls, agreements, gs=gs)
        ev.update(replay_setup_ms=gs["setup_ms"], replay_capture_ms=gs["capture_ms"], replay_reserved_after_capture_mib=reserved,
                  replay_mask_transfer_ms=transfer_ms, replay_verification=vr)
    finally:
        release_all()
    return ev, None

print("harness defined:", len(METHODS), "methods per benchmark, reference", method_label(*REFERENCE_COMBINATION),
      f"| cache falls back to at most {CACHE_GROUPS_MAX} run aligned groups on out of memory | cache replay and {len(DIRECT_REPLAY_MENU)} masked attention replays: {GRAPH_WARMUPS} graph warmups, one capture")

harness defined: 17 methods per benchmark, reference math@1024 | cache falls back to at most 8 run aligned groups on out of memory | cache replay and 6 masked attention replays: 3 graph warmups, one capture


## Cell 5. The suite

Runs the 18 benchmarks in `execution_order`. Per benchmark: load the input with its hash checks, time shared preparation, cache own preparation and masked attention own preparation five times each, build the training step model and optimizer from the file for training step rows, run the reference evaluation, then the cache, then the other eight direct combinations in menu order, and write the `benchmarks` row. Rows are appended to the store as they are produced and the five datasets with the metadata are saved to `results\section7_benchmarks_<MACHINE>.pkl.lz4` after preparation and after every evaluation. Rerunning the cell resumes. Benchmarks with status completed or reference unavailable are kept, any other is rerun whole. Every row of a benchmark is identified by its evaluation id prefix, so a partial benchmark is removed completely before it reruns.

The cache's level plan is on the GPU only during cache own preparation, which times its transfer, and during the cache evaluation, where its transfer is repeated untimed inside the handled boundary. No direct combination is measured with it resident. The training step model and optimizer stay resident for every method.

A regenerated input that differs from its file, or a file whose hash check fails, stops the suite. Before stopping, the benchmark's planned rows are saved with the mismatch in their detail, evaluations as error, preparation and agreement rows as not run, the benchmark as reference unavailable, and such a benchmark reruns on resume once the input is corrected. A reference that is not finite, whose capture fails, or whose training step model cannot be built leaves the benchmark reference unavailable. Its other nine evaluations get status reference unavailable at stage not started and agreement rows of status not run. A cache own preparation or plan transfer failure gives the cache evaluation that status at stage agreement. A masked attention own preparation failure gives every masked attention evaluation that status.

The suite budget is 10 h of operational wall clock on the H100, accumulated across resumes in the metadata and checked between benchmarks, so a benchmark started under the budget runs to its end. A benchmark not started after the budget is spent keeps its rows with status not run budget exhausted, preparation and agreement rows included. An interruption with the budget spent keeps the rows produced, the in flight evaluation removed, the evaluations not started as not run budget exhausted, and gives the benchmark status incomplete budget exhausted. An interruption under the budget, on either machine, drops the partial benchmark, which reruns on resume. The 2080 has no budget.

Metadata records the schema version, the manifest and its hash, the five library file hashes, the machine environment, the tolerances, the timing boundaries, the ceiling, the budget and its between benchmark rule, the agreement and capture call rule, the creation time and `mock = False`.

In [10]:
# Estimated run time: 2080, about 2.5 hours for the full suite (1.7 h before the six masked attention replays); H100, about 1.5 hours. Resumable, saves after every evaluation.
import json
RESULTS_PATH = os.path.join(RESULT_DIR, f"section7_benchmarks_{MACHINE}.pkl.lz4")
SCHEMA_VERSION = "Frozen Benchmark Plan v3, generated cache preparation and per method preparation charges, 2026-11-08"
BUDGET_H = SUITE_BUDGET_H if MACHINE == "H100" else None
KEEP_STATUSES = {"completed", "reference unavailable"}
MISMATCH = "frozen input mismatch"
PREP_COMPONENTS = ["input generation", "subset materialization", "cache own preparation", "masked attention own preparation"]
BENCHMARK_FIELDS = ["benchmark_id", "benchmark_input_id", "benchmark_input_sha256", "machine", "benchmark_type",
                    "subset_selection", "experimental_role", "n", "T", "budget_M", "requested_run_length_lambda",
                    "P", "coverage", "F", "N", "reuse_ratio", "full_power_set_ratio", "D", "C", "d_k", "d_v",
                    "precision", "seed", "sampler_key", "execution_order", "paper_destination"]
REF_LABEL = method_label(*REFERENCE_COMBINATION)

class FrozenInputMismatch(RuntimeError):
    pass

manifest_records = json.loads(manifest.to_json(orient="records"))
manifest_sha = hashlib.sha256(json.dumps(manifest_records, sort_keys=True).encode()).hexdigest()

def empty_store():
    return {"benchmarks": [], "method_evaluations": [], "execution_calls": [], "agreement_checks": [],
            "preparation_measurements": [], "metadata": {
                "schema_version": SCHEMA_VERSION, "machine": MACHINE, "manifest": manifest_records,
                "manifest_sha256": manifest_sha, "implementation_commit": IMPLEMENTATION_COMMIT,
                "implementation_file_sha256": IMPLEMENTATION_FILE_SHA256, "machine_environment": MACHINE_ENVIRONMENT,
                "tolerances": {"output": TOL32_OUT, "gradient_relative": TOL32_GRAD, "gradient_zero": TOL32_ZERO,
                               "loss": TOL_LOSS, "parameter_gradient_relative": TOL_GRAD, "parameter_gradient_zero": TOL_ZERO},
                "timing_boundaries": {
                    "execution": "synchronized wall clock of one call, cache unchunked unless its unchunked evaluation ran out of memory, then over the fewest run aligned groups that fit, cache replay as one replay of the captured graph (attention: the input copies into the graph's tensors plus the replay; training step: the replay, state restored in place before the call), direct chunk loop with each mask chunk transferred to the GPU inside the call, training step from restored state through the optimizer update",
                    "cache replay setup": "graph warmups and one capture, timed once per evaluation and recorded in the evaluation row, not in preparation_measurements",
                    "masked attention replay": "each masked attention combination's call captured once in a CUDA graph and replayed, as cache replay; its whole P x n mask is moved to the GPU once before the graph warmups, timed and recorded as replay_mask_transfer_ms, not in execution, so each replay reads mask rows already on the GPU; outputs stay referenced in the graph; gathered attention builds its indices on the host inside the call and has no replay",
                    "input generation": "Q, K, V and W from the seed on attention rows, nothing on training step rows, charged to every method",
                    "subset materialization": "the subset list regenerated from the key, charged to masked attention and gathered attention, and to the cache only on fixed size run and individual sampler rows, where its trie is built from the list",
                    "cache own preparation": "trie, level plan and its transfer to the GPU, synchronized; on run sampler and full power set rows the trie is generated from the rank intervals of the row's descriptor by the successor rule, on the other rows it is built from the subset list; each preparation row carries charged_to, the methods that pay for it",
                    "masked attention own preparation": "the P x n boolean mask on the CPU, no transfer",
                    "gathered attention preparation": "none by protocol, indices built inside execution",
                    "plan residency": "the level plan is on the GPU only during cache own preparation and the cache and cache replay evaluations",
                    "graph residency": "the captured graph and its memory pool exist only during its own replay evaluation, cache replay or masked attention replay",
                    "memory": "peak allocated and peak reserved per call, counters reset by the cleanup before each call; replay intermediates live in the graph's pool, so reserved is the comparable figure"},
                "agreement_rule": "agreement call = the method as timed, nothing collected, wall clock decides the ceiling; capture call = the method again untimed with outputs copied to the host, compared with the reference; training step capture omits the optimizer update, except the replays (cache replay and masked attention replay), whose captured step includes it and whose loss and gradients are read before the update; no capture for a nonreference method above the ceiling",
                "replay_verification_rule": f"before any timed call, untimed: {REPLAY_CHECKS} replays compared with the first (attention: output and gradients by the float32 rules; training step: loss, gradients, the parameter update, both AdamW moment buffers and the step counters, gradients and buffers by relative error <= {TOL_GRAD}), and for a training step the first replay compared the same way with the eager step of the same method from the same snapshot; any failure ends the evaluation as failed agreement, recorded in replay_verification",
                "budget_rule": "operational wall clock accumulated across resumes, checked between benchmarks",
                "ceiling_ms": CEILING_MS, "suite_budget_h": BUDGET_H, "warmups": WARMUPS, "timed_repeats": TIMED_REPEATS,
                "preparation_repeats": PREP_REPEATS, "graph_warmups": GRAPH_WARMUPS, "suite_elapsed_s": 0.0, "created": time.strftime("%Y-%m-%d %H:%M:%S"),
                "mock": False}}

def save_store(store, elapsed):
    store["metadata"]["suite_elapsed_s"] = elapsed
    out = {k: pd.DataFrame(store[k]) for k in ("benchmarks", "method_evaluations", "execution_calls",
                                                 "agreement_checks", "preparation_measurements")}
    out["metadata"] = store["metadata"]
    save_to_pickle(out, RESULTS_PATH, lz4_compression=True)

def load_store():
    if not os.path.exists(RESULTS_PATH):
        return empty_store()
    saved = load_from_pickle(RESULTS_PATH)
    assert saved["metadata"]["manifest_sha256"] == manifest_sha, "saved results were produced under a different manifest"
    assert saved["metadata"]["implementation_file_sha256"] == IMPLEMENTATION_FILE_SHA256, "library files changed since the saved results"
    store = {k: saved[k].to_dict("records") for k in ("benchmarks", "method_evaluations", "execution_calls",
                                                       "agreement_checks", "preparation_measurements")}
    store["metadata"] = saved["metadata"]
    return store

def belongs(eid, bid):
    return isinstance(eid, str) and eid.startswith(bid + "#")

def drop_benchmark(store, bid):
    """Every row of the benchmark, identified by the evaluation id prefix, never through the evaluation rows."""
    store["benchmarks"] = [b for b in store["benchmarks"] if b["benchmark_id"] != bid]
    store["method_evaluations"] = [e for e in store["method_evaluations"] if not belongs(e["evaluation_id"], bid)]
    store["execution_calls"] = [c for c in store["execution_calls"] if not belongs(c["evaluation_id"], bid)]
    store["agreement_checks"] = [a for a in store["agreement_checks"] if not belongs(a["evaluation_id"], bid)]
    store["preparation_measurements"] = [p for p in store["preparation_measurements"] if p["benchmark_id"] != bid]

def drop_inflight(store, bid):
    """Calls and agreement rows of evaluations that have no evaluation row yet."""
    have = {e["evaluation_id"] for e in store["method_evaluations"] if belongs(e["evaluation_id"], bid)}
    store["execution_calls"] = [c for c in store["execution_calls"] if not (belongs(c["evaluation_id"], bid) and c["evaluation_id"] not in have)]
    store["agreement_checks"] = [a for a in store["agreement_checks"] if not (belongs(a["evaluation_id"], bid) and a["evaluation_id"] not in have)]
    return have

def benchmark_row(row, status):
    b = {k: row[k] for k in BENCHMARK_FIELDS}
    for k in ("n", "budget_M", "P", "F", "N", "D", "C", "d_k", "d_v", "seed", "execution_order"):
        b[k] = int(b[k])
    b["requested_run_length_lambda"] = None if pd.isna(b["requested_run_length_lambda"]) else int(b["requested_run_length_lambda"])
    b["benchmark_status"], b["reference_evaluation_id"] = status, f"{row.benchmark_id}#{REF_LABEL}"
    return b

def method_order():
    """Reference first, then the cache, then cache replay, then the remaining eager direct combinations in menu order,
    then the masked attention replays in menu order."""
    rest = [m for m in METHODS if m != REFERENCE_COMBINATION and m[0] not in CACHE_METHODS and m[0] != DIRECT_REPLAY]
    return [REFERENCE_COMBINATION, (CACHE, None, None), (REPLAY, None, None)] + rest + DIRECT_REPLAY_MENU

def unrun_rows(row, status, stage="not started", detail=None, skip=()):
    """Evaluation and agreement rows for methods that never run. Agreement status is not run."""
    evs, ags = [], []
    ref_id = f"{row.benchmark_id}#{REF_LABEL}"
    for m in method_order():
        if m in skip:
            continue
        ev = new_evaluation_row(row, *m)
        ev["status"], ev["status_stage"], ev["detail"] = status, stage, detail
        evs.append(ev)
        if not ev["is_reference"]:
            ags.append(new_agreement_row(ev, ref_id, "not run", None, False, detail))
    return evs, ags

def unrun_preparation_rows(row, status="not run", detail=None, skip=()):
    return [{"benchmark_id": row.benchmark_id, "preparation_component": comp, "repeat_index": i, "status": status,
             "wall_ms": None, "detail": detail, "charged_to": None}
            for comp in PREP_COMPONENTS if comp not in skip for i in range(1, PREP_REPEATS + 1)]

def prep_failure(preps, comp):
    bad = [p for p in preps if p["preparation_component"] == comp and p["status"] != "completed"]
    return (bad[-1]["status"], bad[-1]["detail"]) if bad else (None, None)

def record_mismatch(row, store, clock, detail, preps_done=()):
    """The benchmark's planned rows with the mismatch in their detail, then the suite stops."""
    bid = row.benchmark_id
    store["preparation_measurements"].extend(unrun_preparation_rows(row, "not run", detail, skip=preps_done))
    e, a = unrun_rows(row, "error", "not started", detail)
    store["method_evaluations"].extend(e); store["agreement_checks"].extend(a)
    store["benchmarks"].append(benchmark_row(row, "reference unavailable"))
    save_store(store, clock())
    raise FrozenInputMismatch(f"{bid}: {detail}")

def run_benchmark(row, store, clock):
    """One benchmark. Every planned row lands in the store as it is produced, saved after preparation and
    after every evaluation. Returns the benchmark status."""
    bid = row.benchmark_id
    ref_id = f"{bid}#{REF_LABEL}"
    try:
        ctx = load_input(row)
    except AssertionError as exc:
        record_mismatch(row, store, clock, f"{MISMATCH}: {exc}")
    except Exception as exc:
        status, detail = classify(exc)
        store["preparation_measurements"].extend(unrun_preparation_rows(row, "not run", f"input not loaded: {detail}"))
        e, a = unrun_rows(row, "error", "not started", f"input not loaded: {detail}")
        store["method_evaluations"].extend(e); store["agreement_checks"].extend(a)
        store["benchmarks"].append(benchmark_row(row, "reference unavailable"))
        save_store(store, clock())
        return "reference unavailable"
    step = ctx["step"]

    # protocol step 1, preparation
    preps, plan_np, mask_cpu = [], None, None
    for comp in PREP_COMPONENTS:
        for i in range(1, PREP_REPEATS + 1):
            p = {"benchmark_id": bid, "preparation_component": comp, "repeat_index": i, "status": "completed",
                 "wall_ms": None, "detail": None, "charged_to": charged_to(ctx, comp)}
            try:
                if comp == "input generation":
                    p["wall_ms"] = input_generation(ctx, i)
                elif comp == "subset materialization":
                    p["wall_ms"] = subset_materialization(ctx, i)
                elif comp == "cache own preparation":
                    plan_np = None
                    clean()
                    plan_dev, p["wall_ms"] = cache_preparation(ctx)
                    del plan_dev
                    clean()
                    if i == PREP_REPEATS:
                        plan_np = forest_level_plan_np([cache_trie(ctx)], [0])
                else:
                    mask_cpu = None
                    mask_cpu, p["wall_ms"] = masked_preparation(ctx)
            except AssertionError as exc:
                p["status"], p["detail"] = "error", f"{MISMATCH}: {exc}"
                preps.append(p)
                store["preparation_measurements"].extend(preps)
                done_comps = {q["preparation_component"] for q in preps}
                record_mismatch(row, store, clock, p["detail"], preps_done=done_comps)
            except Exception as exc:
                p["status"], p["detail"] = classify(exc)
            preps.append(p)
    store["preparation_measurements"].extend(preps)
    save_store(store, clock())
    cache_fail = prep_failure(preps, "cache own preparation")
    masked_fail = prep_failure(preps, "masked attention own preparation")

    # training step model and optimizer from the file
    s = None
    if step:
        try:
            clean()
            s = step_models(ctx)
        except Exception as exc:
            status, detail = classify(exc)
            e, a = unrun_rows(row, "reference unavailable", "not started", f"training step model: {detail}", skip=[REFERENCE_COMBINATION])
            ref_ev = new_evaluation_row(row, *REFERENCE_COMBINATION)
            ref_ev.update(status=status, status_stage="reference", detail=f"training step model: {detail}")
            store["method_evaluations"].extend([ref_ev] + e); store["agreement_checks"].extend(a)
            store["benchmarks"].append(benchmark_row(row, "reference unavailable"))
            save_store(store, clock())
            return "reference unavailable"

    # reference, cache, cache replay, the other eight
    ref, ref_ok = None, True
    for k, (method, backend, chunk) in enumerate(method_order()):
        if k > 0 and not ref_ok:
            break
        label = method_label(method, backend, chunk)
        failed = cache_fail if method in CACHE_METHODS else (masked_fail if method in ("masked attention", DIRECT_REPLAY) else (None, None))
        plan = None
        if failed[0] is None and method in CACHE_METHODS:
            try:
                plan = plan_to_device(plan_np, DEVICE)
            except Exception as exc:
                st, dt = classify(exc)
                failed = (st, f"plan transfer: {dt}")
        if failed[0] is not None:
            ev = new_evaluation_row(row, method, backend, chunk)
            comp = "cache own preparation" if method in CACHE_METHODS else "masked attention own preparation"
            ev.update(status=failed[0], status_stage="reference" if ev["is_reference"] else "agreement", detail=f"{comp}: {failed[1]}")
            store["method_evaluations"].append(ev)
            if ev["is_reference"]:
                ref_ok = False
            else:
                store["agreement_checks"].append(new_agreement_row(ev, ref_id, failed[0], None, False, ev["detail"]))
        else:
            if method == CACHE:
                ev, ref_out = evaluate_cache(ctx, row, plan, mask_cpu, ref, s, store["execution_calls"], store["agreement_checks"])
            elif method in REPLAY_METHODS:
                ev, ref_out = evaluate_replay(ctx, row, method, backend, chunk, plan, mask_cpu, ref, s,
                                              store["execution_calls"], store["agreement_checks"])
            else:
                ev, ref_out = evaluate_method(ctx, row, method, backend, chunk, plan, mask_cpu, ref, s,
                                              store["execution_calls"], store["agreement_checks"])
            store["method_evaluations"].append(ev)
            if ev["is_reference"]:
                if ref_out is None:
                    ref_ok = False
                else:
                    ref = ref_out
        del plan
        clean()
        save_store(store, clock())
        med = ev["execution_median_ms"]
        groups_note = f"  ({ev['cache_groups']} groups)" if (ev.get("cache_groups") or 1) > 1 else ""
        print(f"    {label:20s} {ev['status']:26s} {'' if med is None else f'{med:10.1f} ms'}{groups_note}", flush=True)
    if ref_ok:
        bstatus = "completed"
    else:
        e, a = unrun_rows(row, "reference unavailable", "not started", "reference unavailable", skip=[REFERENCE_COMBINATION])
        store["method_evaluations"].extend(e); store["agreement_checks"].extend(a)
        bstatus = "reference unavailable"
    store["benchmarks"].append(benchmark_row(row, bstatus))
    save_store(store, clock())
    del ctx, plan_np, mask_cpu, s, ref
    clean()
    return bstatus

# ----- the loop -----
store = load_store()
EXEC_ORDER[0] = max([c["execution_order"] for c in store["execution_calls"] if c["execution_order"] is not None], default=0)
done = {b["benchmark_id"]: b["benchmark_status"] for b in store["benchmarks"]}
mismatched = {e["benchmark_id"] for e in store["method_evaluations"] if isinstance(e.get("detail"), str) and e["detail"].startswith(MISMATCH)}
elapsed0 = float(store["metadata"].get("suite_elapsed_s", 0.0))
t_suite = time.perf_counter()
clock = lambda: elapsed0 + (time.perf_counter() - t_suite)
kept = [b for b, v in done.items() if v in KEEP_STATUSES and b not in mismatched]
print(f"{MACHINE}: {len(kept)} of 18 benchmarks kept from {RESULTS_PATH}, {elapsed0 / 3600:.2f} h operational so far")
for _, row in manifest.iterrows():
    bid = row.benchmark_id
    if bid in kept:
        continue
    drop_benchmark(store, bid)
    if BUDGET_H is not None and clock() >= BUDGET_H * 3600:
        store["preparation_measurements"].extend(unrun_preparation_rows(row, "not run", "budget exhausted"))
        e, a = unrun_rows(row, "not run budget exhausted", "not started", "budget exhausted")
        store["method_evaluations"].extend(e); store["agreement_checks"].extend(a)
        store["benchmarks"].append(benchmark_row(row, "not run budget exhausted"))
        save_store(store, clock())
        print(f"{row.execution_order:2d} {bid:52s} not run budget exhausted")
        continue
    print(f"{row.execution_order:2d} {bid:52s} {clock() / 60:7.1f} min elapsed", flush=True)
    t_b = time.perf_counter()
    try:
        bstatus = run_benchmark(row, store, clock)
    except KeyboardInterrupt:
        if BUDGET_H is not None and clock() >= BUDGET_H * 3600:
            have = drop_inflight(store, bid)
            e, a = unrun_rows(row, "not run budget exhausted", "not started", "interrupted, budget exhausted")
            store["method_evaluations"].extend([x for x in e if x["evaluation_id"] not in have])
            store["agreement_checks"].extend([x for x in a if x["evaluation_id"] not in have])
            comps = {p["preparation_component"] for p in store["preparation_measurements"] if p["benchmark_id"] == bid}
            store["preparation_measurements"].extend(unrun_preparation_rows(row, "not run", "interrupted, budget exhausted", skip=comps))
            store["benchmarks"].append(benchmark_row(row, "incomplete budget exhausted"))
            save_store(store, clock())
            print(f"interrupted during {bid} with the budget spent, saved as incomplete budget exhausted")
        else:
            drop_benchmark(store, bid)
            save_store(store, clock())
            print(f"interrupted during {bid}, partial rows dropped, the benchmark reruns on resume")
        raise
    print(f"   {bstatus}, {(time.perf_counter() - t_b) / 60:6.1f} min, saved", flush=True)

summary = pd.DataFrame(store["method_evaluations"])
print(f"{MACHINE}: {len(store['benchmarks'])} benchmarks, {len(summary)} evaluations, {len(store['execution_calls'])} execution calls, "
      f"{len(store['agreement_checks'])} agreement checks, {len(store['preparation_measurements'])} preparation measurements, "
      f"{store['metadata']['suite_elapsed_s'] / 3600:.2f} h operational")
summary.groupby(["benchmark_id", "status"]).size().unstack(fill_value=0)

2080: 0 of 18 benchmarks kept from E:\apriori-ml-v3\results\section7_benchmarks_2080.pkl.lz4, 0.00 h operational so far
 1 v2_small_record_boundary_attention_n14_M16383@2080       0.0 min elapsed
    math@1024            timed                            22.6 ms
    cache                timed                            40.1 ms
    cache replay         timed                            28.5 ms
    default@all          timed                            81.2 ms
    default@4096         timed                            25.4 ms
    default@1024         timed                            39.2 ms
    math@all             timed                            31.7 ms
    math@4096            timed                            98.7 ms
    gathered@all         timed                            87.8 ms
    gathered@4096        timed                            77.3 ms
    gathered@1024        timed                           142.6 ms
    default@all replay   timed                            30.8 ms
    default@

status,ceiling exceeded,out of memory,timed
benchmark_id,,,
v2_fixed_size_run_comparison_attention_n300_M64000@2080,0,3,14
v2_fixed_size_run_comparison_attention_n496_M64000@2080,0,3,14
v2_limited_sharing_control_attention_n300_M64000@2080,0,4,13
v2_limited_sharing_control_attention_n496_M64000@2080,0,5,12
v2_main_comparison_attention_n300_M128000@2080,0,5,12
v2_main_comparison_attention_n300_M256000@2080,0,7,10
v2_main_comparison_attention_n300_M512000@2080,0,11,6
v2_main_comparison_attention_n300_M64000@2080,0,3,14
v2_main_comparison_attention_n496_M128000@2080,0,5,12


## Cell 6. Run trie generator equivalence

The v3 preparation study. The canonical builder reads a sorted list of subsets and finds each subset's shared prefix by comparing it with its predecessor. The generator reads no subsets. It takes the row's run descriptor, `n`, `M`, the run length and the sampler key, recomputes the offset with the sampler's own helper `powerset_run_offset`, turns the runs into nonwrapping rank intervals with `powerset_run_intervals`, unranks the first subset of each interval, and walks the rest by the successor rule in `generate_run_trie`. One stack is carried across the intervals in rank order. At an interval boundary the stack is cut to the prefix shared with the previous subset. The node insertion order equals the builder's over the sorted family, so the four arrays `pos`, `parent`, `depth` and `terminals` are identical.

This cell establishes two facts and takes the first timing.

- Fact 1. The intervals recomputed from the row's key describe exactly the subsets stored in the frozen input, in stored order.
- Fact 2. The generated arrays equal the builder's arrays.

Part A enumerates every offset for n = 1 to 6, every budget from 1 to T + 5 and five run lengths, against a test oracle that repeats the sampler's loop for an explicit offset. That covers wrapped runs, adjacent runs, gaps with shared prefixes, n = 1 and M ≥ T, which the frozen inputs alone do not exercise. It also checks the hashed path for three keys so the shared offset helper is exercised. Part B loads the 14 frozen inputs the generator covers, 12 run sampler rows and the two full power set rows, asserts both facts and times three things on the CPU. `sampler_ms` is `generate_subsets(row)`, the regeneration shared preparation times today. `builder_ms` is the builder on the stored tuples. `generator_ms` is the offset draw, the intervals and the walk together, the descriptor to trie path with no subset list. The four fixed size and individual sampler rows have no run structure and keep the builder. Level plan construction and transfer are not timed here. They are the same work for either trie and are measured in the suite.

Nothing in this cell changes the suite. The suite still prepares the cache with the builder until this study decides otherwise.

In [4]:
# Estimated run time: about 5 minutes on the 2080 host (the sampler regeneration for the four M = 512,000 rows is most of it).
# Cell 6. Run trie generator equivalence. Requires Cells 1, 2 (verify mode) and 4.
from set_attention_cache import (powerset_run_offset, powerset_run_intervals, generate_run_trie,
                                 powerset_run_ranks, unrank_lex_subset, canonical_successor)

def reference_family(n, M, run_len, offset):
    """Test oracle: the sampler's own loop for an explicit offset (sample_subsets_powerset_runs draws the offset itself)."""
    T = powerset_total(n)
    if M >= T:
        return [unrank_lex_subset(r, n) for r in range(T)]
    out = []
    for start, L in powerset_run_ranks(n, M, run_len, offset):
        S = unrank_lex_subset(start, n)
        for _ in range(L):
            out.append(S)
            S = canonical_successor(S, n)
            if S is None:
                S = (0,)
    out.sort()
    return out

def materialize_intervals(n, intervals):
    """Test only: the subsets an interval list names, by unranking each start and stepping the successor."""
    out = []
    for start, length in intervals:
        S = unrank_lex_subset(start, n)
        for _ in range(length):
            out.append(S)
            S = canonical_successor(S, n)
    return out

def tries_equal(a, b):
    return all(np.array_equal(a[k], b[k]) for k in ("pos", "parent", "depth", "terminals"))

# ----- Part A: enumerated cases, every offset -----
t0 = time.perf_counter()
cases = wraps = adjacent = 0
for n in range(1, 7):
    T = powerset_total(n)
    for M in list(range(1, T)) + [T, T + 5]:
        for run_len in (1, 2, 3, 5, 8):
            for offset in (range(T) if M < T else [0]):
                fam = reference_family(n, M, run_len, offset)
                iv = powerset_run_intervals(n, M, run_len, offset)
                assert materialize_intervals(n, iv) == fam, (n, M, run_len, offset)
                assert tries_equal(generate_run_trie(n, iv), build_prefix_trie_canonical(fam)), (n, M, run_len, offset)
                cases += 1
                if M < T:
                    wraps += len(iv) > len(powerset_run_ranks(n, M, run_len, offset))
                    adjacent += any(a + la == b for (a, la), (b, _) in zip(iv, iv[1:]))
for key in ("a", "pvcb_n6_set0", "v2_main_comparison_attention_n300_M64000"):     # the hashed path through the shared helper
    for n, M, run_len in ((6, 20, 4), (10, 300, 8), (12, 1000, 120)):
        fam = sample_subsets_powerset_runs(n, key, M, run_len, seed=SEED, epoch=0)
        iv = powerset_run_intervals(n, M, run_len, powerset_run_offset(n, key, seed=SEED, epoch=0))
        assert materialize_intervals(n, iv) == fam, (key, n, M, run_len)
        assert tries_equal(generate_run_trie(n, iv), build_prefix_trie_canonical(fam)), (key, n, M, run_len)
print(f"Part A: {cases} enumerated cases pass, {wraps} with a wrapped run, {adjacent} with adjacent runs, "
      f"{time.perf_counter() - t0:.1f} s")

# ----- Part B: the frozen inputs -----
row_intervals = run_intervals                              # the harness function, Cell 4

def best_of(fn, repeats=3):
    best = None
    for _ in range(repeats):
        t = time.perf_counter()
        out = fn()
        ms = 1000 * (time.perf_counter() - t)
        best = ms if best is None else min(best, ms)
    return out, best

rows = []
for _, row in manifest.iterrows():
    rec = load_from_pickle(input_path(row.benchmark_input_id))
    n, sel = int(rec["n"]), rec["subset_selection"]
    subs = tuples_from_flat(rec["subset_indptr"], rec["subset_flat"])
    iv = row_intervals(rec)
    if iv is None:
        rows.append(dict(benchmark_input_id=row.benchmark_input_id, subset_selection=sel, n=n, P=len(subs),
                         note="builder retained, no run structure"))
        continue
    family_equal = materialize_intervals(n, iv) == subs
    _, sampler_ms = best_of(lambda: generate_subsets(row), 1)
    trie_b, builder_ms = best_of(lambda: build_prefix_trie_canonical(subs))
    trie_g, generator_ms = best_of(lambda: generate_run_trie(n, row_intervals(rec)))   # offset, intervals and walk inside the clock
    trie_equal = tries_equal(trie_g, trie_b)
    assert family_equal and trie_equal, row.benchmark_input_id
    rows.append(dict(benchmark_input_id=row.benchmark_input_id, subset_selection=sel, n=n, P=len(subs),
                     N=len(trie_b["pos"]), intervals=len(iv), family_equal=family_equal, trie_equal=trie_equal,
                     sampler_ms=round(sampler_ms, 1), builder_ms=round(builder_ms, 1), generator_ms=round(generator_ms, 1),
                     generator_over_builder=round(generator_ms / builder_ms, 3), note=""))
    del rec, subs, trie_b, trie_g
equivalence = pd.DataFrame(rows)
checked = equivalence[equivalence.note == ""]
assert len(checked) == 14 and checked.family_equal.all() and checked.trie_equal.all()
print(f"Part B: {len(checked)} frozen inputs, intervals equal the stored subsets and the four trie arrays equal the builder's")
equivalence

Part A: 25530 enumerated cases pass, 6713 with a wrapped run, 5822 with adjacent runs, 3.6 s
Part B: 14 frozen inputs, intervals equal the stored subsets and the four trie arrays equal the builder's


,benchmark_input_id,subset_selection,n,P,N,intervals,family_equal,trie_equal,sampler_ms,builder_ms,generator_ms,generator_over_builder,note
0,v2_small_record_boundary_attention_n14_M16383,full power set,14,16383,16383.0,1.0,True,True,53.5,20.9,9.3,0.447,
1,v2_main_comparison_attention_n300_M64000,run sampler,300,63000,66107.0,21.0,True,True,1371.8,574.1,41.9,0.073,
2,v2_fixed_size_run_comparison_attention_n300_M6...,fixed size run sampler,300,64000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"builder retained, no run structure"
3,v2_limited_sharing_control_attention_n300_M64000,individual sampler,300,64000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"builder retained, no run structure"
4,v2_main_comparison_attention_n496_M64000,run sampler,496,59520,62468.0,12.0,True,True,2089.7,888.1,40.6,0.046,
5,v2_fixed_size_run_comparison_attention_n496_M6...,fixed size run sampler,496,64000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"builder retained, no run structure"
6,v2_limited_sharing_control_attention_n496_M64000,individual sampler,496,64000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"builder retained, no run structure"
7,v2_main_comparison_step_n300_M64000,run sampler,300,63000,66105.0,21.0,True,True,1390.0,581.2,42.2,0.073,
8,v2_main_comparison_step_n496_M64000,run sampler,496,59520,62470.0,12.0,True,True,2104.8,873.6,38.1,0.044,
9,v2_small_record_boundary_attention_n16_M65535,full power set,16,65535,65535.0,1.0,True,True,252.6,88.4,37.7,0.426,


## Cell 7. Preparation components on the two largest rows

Every component of protocol step 1 on its own clock, for the n = 300 and n = 496 attention rows at M = 512,000. Five repeats each, minimum and median reported. Clocks follow the harness. CPU components use the wall clock. The transfer is synchronized on the GPU.

- `sampler`. `generate_subsets(row)`, the subset list regenerated from the key. Needed today by both methods. Not needed by the cache once the trie is generated from the descriptor.
- `builder`. `build_prefix_trie_canonical` on the stored tuples. Today's trie.
- `generator`. Offset, intervals and `generate_run_trie`. The descriptor to trie path.
- `plan`. `forest_level_plan_np` on the trie. Unchanged by the generator.
- `transfer`. `plan_to_device` to the GPU, synchronized. Unchanged by the generator.
- `mask`. `subset_mask` on the stored tuples, the P × n boolean mask. Masked attention's own preparation today.

The summary charges each method for what it needs. Cache today is sampler, builder, plan and transfer. Cache generated is generator, plan and transfer. Masked attention is sampler and mask, as today. Gathered attention has no own preparation and is not shown. This cell measures preparation only. It says nothing about execution time or memory.

In [5]:
# Estimated run time: about 4 minutes on the 2080 (the sampler at M = 512,000 is 11 to 18 s per call, five calls per row).
# Cell 7. Preparation components on the two largest rows. Requires Cells 1, 2, 4 and 6.
C7_ROWS = ["v2_main_comparison_attention_n300_M512000", "v2_main_comparison_attention_n496_M512000"]
C7_REPEATS = PREP_REPEATS                                   # 5, as the suite

def repeat_ms(fn, repeats, gpu=False):
    """Minimum and median wall clock over repeats. gpu=True synchronizes before and after each call."""
    ms = []
    for _ in range(repeats):
        if gpu:
            out, t = timed(fn)
        else:
            t0 = time.perf_counter()
            out = fn()
            t = 1000 * (time.perf_counter() - t0)
        ms.append(t)
        del out
    return min(ms), float(np.median(ms))

components = []
for input_id in C7_ROWS:
    row = manifest[manifest.benchmark_input_id == input_id].iloc[0]
    rec = load_from_pickle(input_path(input_id))
    n = int(rec["n"])
    subs = tuples_from_flat(rec["subset_indptr"], rec["subset_flat"])
    trie = generate_run_trie(n, row_intervals(rec))
    assert tries_equal(trie, build_prefix_trie_canonical(subs)), input_id
    plan_np = forest_level_plan_np([trie], [0])
    clean()
    timings = {
        "sampler":   repeat_ms(lambda: generate_subsets(row), C7_REPEATS),
        "builder":   repeat_ms(lambda: build_prefix_trie_canonical(subs), C7_REPEATS),
        "generator": repeat_ms(lambda: generate_run_trie(n, row_intervals(rec)), C7_REPEATS),
        "plan":      repeat_ms(lambda: forest_level_plan_np([trie], [0]), C7_REPEATS),
        "transfer":  repeat_ms(lambda: plan_to_device(plan_np, DEVICE), C7_REPEATS, gpu=True),
        "mask":      repeat_ms(lambda: subset_mask(subs, n, "cpu"), C7_REPEATS),
    }
    for comp, (mn, md) in timings.items():
        components.append(dict(benchmark_input_id=input_id, n=n, P=len(subs), N=len(trie["pos"]), component=comp,
                               min_ms=round(mn, 1), median_ms=round(md, 1), repeats=C7_REPEATS))
    del rec, subs, trie, plan_np
    clean()

prep_components = pd.DataFrame(components)

def charged(df, parts):
    return df[df.component.isin(parts)].median_ms.sum()

summary = []
for input_id in C7_ROWS:
    d = prep_components[prep_components.benchmark_input_id == input_id]
    today = charged(d, ["sampler", "builder", "plan", "transfer"])
    generated = charged(d, ["generator", "plan", "transfer"])
    summary.append(dict(benchmark_input_id=input_id,
                        cache_today_ms=round(today, 1), cache_generated_ms=round(generated, 1),
                        cache_ratio=round(today / generated, 2),
                        builder_share_of_today=round(charged(d, ["builder"]) / today, 3),
                        plan_transfer_share_of_generated=round(charged(d, ["plan", "transfer"]) / generated, 3),
                        masked_ms=round(charged(d, ["sampler", "mask"]), 1)))
prep_summary = pd.DataFrame(summary)
print(prep_components.pivot(index="component", columns="benchmark_input_id", values="median_ms")
      .loc[["sampler", "builder", "generator", "plan", "transfer", "mask"]])
prep_summary

benchmark_input_id  v2_main_comparison_attention_n300_M512000  \
component                                                       
sampler                                               10934.2   
builder                                                4721.3   
generator                                               378.1   
plan                                                    100.6   
transfer                                                 26.3   
mask                                                   6316.7   

benchmark_input_id  v2_main_comparison_attention_n496_M512000  
component                                                      
sampler                                               17526.8  
builder                                                7522.5  
generator                                               407.1  
plan                                                    100.8  
transfer                                                 27.5  
mask                           

,benchmark_input_id,cache_today_ms,cache_generated_ms,cache_ratio,builder_share_of_today,plan_transfer_share_of_generated,masked_ms
0,v2_main_comparison_attention_n300_M512000,15782.4,505.0,31.25,0.299,0.251,17250.9
1,v2_main_comparison_attention_n496_M512000,25177.6,535.4,47.03,0.299,0.240,27768.6


## Cell 8. Preparation per method from the store

Reads `preparation_measurements` from the 2080 store. Every row carries `charged_to`. For each benchmark the median wall clock per component over the five repeats is shown, then each method's charged preparation, the sum of the medians of the components charged to it. Cache is input generation plus cache own preparation, plus subset materialization only on the fixed size run and individual sampler rows. Masked attention is input generation, subset materialization and the mask. Gathered attention is input generation and subset materialization. The last column is masked attention's charge over the cache's.

In [11]:
# Estimated run time: under 10 seconds.
# Cell 8. Preparation per method from the store. Requires Cells 1 and 4.
store = load_from_pickle(RESULTS_PATH)
prep = pd.DataFrame(store["preparation_measurements"])
assert len(prep) == 360 and (prep.status == "completed").all(), prep.status.value_counts()

med = (prep.groupby(["benchmark_id", "preparation_component"]).wall_ms.median()
       .unstack("preparation_component")[PREP_COMPONENTS])
charges = prep.groupby(["benchmark_id", "preparation_component"]).charged_to.first().unstack("preparation_component")

def charge(bid, method):
    return sum(med.loc[bid, c] for c in PREP_COMPONENTS if method in charges.loc[bid, c])

order = pd.DataFrame(store["benchmarks"]).set_index("benchmark_id").execution_order
out = med.copy()
for m in ("cache", "masked attention", "gathered attention"):
    out[f"{m} charged"] = [charge(b, m) for b in out.index]
out["masked over cache"] = out["masked attention charged"] / out["cache charged"]
out = out.loc[order.sort_values().index].round(1)
out["masked over cache"] = out["masked over cache"].round(2)
prep_by_method = out
prep_by_method

preparation_component,input generation,subset materialization,cache own preparation,masked attention own preparation,cache charged,masked attention charged,gathered attention charged,masked over cache
benchmark_id,,,,,,,,
v2_small_record_boundary_attention_n14_M16383@2080,704.2,69.7,13.7,12.9,717.9,786.9,774.0,1.1
v2_main_comparison_attention_n300_M64000@2080,2651.2,2037.6,57.8,790.7,2709.0,5479.5,4688.8,2.0
v2_fixed_size_run_comparison_attention_n300_M64000@2080,2612.7,6891.0,232.7,151.2,9736.3,9654.9,9503.7,1.0
v2_limited_sharing_control_attention_n300_M64000@2080,2718.4,6242.9,5159.0,802.4,14120.2,9763.6,8961.2,0.7
v2_main_comparison_attention_n496_M64000@2080,2446.3,3089.1,55.7,1209.1,2502.1,6744.6,5535.5,2.7
v2_fixed_size_run_comparison_attention_n496_M64000@2080,2637.0,12614.3,239.5,156.4,15490.9,15407.8,15251.3,1.0
v2_limited_sharing_control_attention_n496_M64000@2080,2648.3,10323.5,8649.0,1303.0,21620.7,14274.8,12971.8,0.7
v2_main_comparison_step_n300_M64000@2080,0.0,2023.7,56.9,790.2,56.9,2813.9,2023.7,49.5
v2_main_comparison_step_n496_M64000@2080,0.0,3116.5,54.7,1205.2,54.7,4321.7,3116.5,79.1
